<a href="https://colab.research.google.com/github/Tanishta15/Ai_Evaluator_IBM/blob/main/All_out_AI_Evaluator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instructions for Using.

1) Segregate the ppts by their themes and upload them in your drive in the MyDrive folder (the root folder).

2) In the folder that you have made, keep a scores folder as well.

3) Final folder structure looks like :

|--MyDrive/

|----|---Theme1/

|----|---Theme2/

|----|---Theme3/

|----|---scores/


4) Click the button on top 'Run all'

In [ ]:
# @title Drive Mounting

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title Install dependencies for Extraction Part
# If running in Colab, also installs the Tesseract binary.
import sys, subprocess, os, pathlib, platform

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *pkgs])

# Core data & OCR
pip_install([
    "pandas", "pyarrow", "fastparquet", "PyMuPDF",
    "pytesseract", "Pillow",
    "python-pptx",          # only for robust image path fallbacks / metadata
])

# IBM DPK
# If your environment uses a different index adjust the names accordingly.
pip_install([
    "docling",
    "docling-ibm-models",
])

# Colab-only: install tesseract executable
if "google.colab" in sys.modules:
    subprocess.check_call(["apt-get", "update", "-y"])
    subprocess.check_call(["apt-get", "install", "-y", "tesseract-ocr"])

print("Installed. Python:", sys.version)

Installed. Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [ ]:
# @title Install dependencies and set up Colab for Certificates

from pathlib import Path
import pandas as pd
import re, json, glob

PARQUET_ROOT = Path("/content/pipeline_output")
DATA_CSV     = Path("/content/drive/MyDrive/hackathon_ppts/data.csv")
RESULTS_CSV  = Path("/content/results/certificate_summary.csv")

# PARQUET_ROOT.mkdir(parents=True, exist_ok=True)
# RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)

print("PARQUET_ROOT:", PARQUET_ROOT)
print("DATA_CSV:", DATA_CSV)

PARQUET_ROOT: /content/pipeline_output
DATA_CSV: /content/drive/MyDrive/hackathon_ppts/data.csv


In [ ]:
# @title env creation

os.environ["WATSONX_URL"] = "https://us-south.ml.cloud.ibm.com"
os.environ["WATSONX_API_KEY"] = "5LMnB69jKazNJp9JyQJZHFjMzKPl2ctBRmPqfjLADLBq"
os.environ["WATSONX_PROJECT_ID"] = "eb3cf1ef-dac5-4516-9102-1b8116badbb7"

In [ ]:
# @title Imports & folder layout
import os, json, re, shutil, traceback, base64
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
from PIL import Image
import pytesseract

from docling.document_converter import DocumentConverter

BASE = Path("/content") if "google.colab" in globals() or "google.colab" in sys.modules else Path(".")
INPUT_DIR = BASE / "input_submissions"
PIPELINE_OUT = BASE / "pipeline_output"

# INPUT_DIR.mkdir(parents=True, exist_ok=True)
# PIPELINE_OUT.mkdir(parents=True, exist_ok=True)

print("INPUT_DIR:", INPUT_DIR)
print("PIPELINE_OUT:", PIPELINE_OUT)

INPUT_DIR: /content/input_submissions
PIPELINE_OUT: /content/pipeline_output


Catergorisation extraction

In [ ]:
# @title Helpers
def safe_text(s: Any) -> str:
    s = "" if s is None else str(s)
    return s.strip()

def save_image_bytes(img: Image.Image, out_path: Path, jpeg_quality: int = 90) -> str:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    # Convert to RGB and save as JPEG to reduce size
    if img.mode != "RGB":
        img = img.convert("RGB")
    img.save(out_path, format="JPEG", quality=jpeg_quality, optimize=True)
    return str(out_path)

def ocr_image_file(path: Path) -> str:
    try:
        with Image.open(path) as im:
            return pytesseract.image_to_string(im)
    except Exception as e:
        return f"[OCR_ERROR] {e}"

In [ ]:
# @title Robust DPK extractor (handles multiple Docling shapes) + OCR (PPT + PDF support)
import io, os, base64, tempfile
import fitz  # PyMuPDF
from pathlib import Path
from pptx import Presentation  # python-pptx fallback
from pptx.enum.shapes import MSO_SHAPE_TYPE
import pandas as pd

def _collect_from_docling_page_blocks(page, sub_out_dir: Path, img_dir: Path, slide_index: int):
    rows = []
    title = ""
    text_chunks = []
    image_paths = []

    for block in getattr(page, "blocks", []):
        btype = (getattr(block, "type", "") or "").lower()
        if btype in ("title", "heading", "header", "h1", "h2") and not title:
            title = safe_text(block.to_text())
        elif btype in ("paragraph", "text", "table", "list"):
            txt = safe_text(block.to_text())
            if txt:
                text_chunks.append(txt)
        elif btype in ("figure", "image", "graphic"):
            pil_img = getattr(block, "pil_image", None)
            if pil_img is None:
                raw = getattr(block, "image_bytes", None)
                if raw:
                    try:
                        pil_img = Image.open(io.BytesIO(raw))
                    except Exception:
                        pil_img = None
            if isinstance(pil_img, Image.Image):
                out_path = img_dir / f"slide_{slide_index}_img_{len(image_paths)+1}.jpg"
                saved = save_image_bytes(pil_img, out_path)
                image_paths.append(saved)

    if not title and text_chunks:
        title = safe_text(text_chunks[0].split("\n")[0])[:120]
    slide_text = "\n".join(text_chunks).strip()

    if image_paths:
        for p in image_paths:
            rows.append({
                "slide_number": slide_index,
                "slide_title": title,
                "slide_content": slide_text,
                "has_image": True,
                "image_path": os.path.relpath(p, sub_out_dir),
                "extracted_content_from_image": "",
            })
    else:
        rows.append({
            "slide_number": slide_index,
            "slide_title": title,
            "slide_content": slide_text,
            "has_image": False,
            "image_path": "",
            "extracted_content_from_image": "",
        })
    return rows

def _collect_from_docling_dict(doc_dict: dict, sub_out_dir: Path, img_dir: Path):
    rows = []
    pages = doc_dict.get("pages") or []
    for idx, page in enumerate(pages, start=1):
        blocks = page.get("blocks") or []
        title = ""
        text_chunks = []
        image_paths = []

        for block in blocks:
            btype = (block.get("type") or "").lower()
            # Text
            if btype in ("title", "heading", "header", "h1", "h2"):
                if not title:
                    title = safe_text(block.get("text") or block.get("content") or "")
            elif btype in ("paragraph", "text", "table", "list"):
                txt = safe_text(block.get("text") or block.get("content") or "")
                if txt:
                    text_chunks.append(txt)
            # Image-like: look for bytes/base64/data fields
            elif btype in ("figure", "image", "graphic"):
                img_b64 = None
                img_info = block.get("image") or {}
                if isinstance(img_info, dict):
                    img_b64 = img_info.get("data") or img_info.get("base64") or None
                    raw_bytes = img_info.get("bytes")
                else:
                    img_b64 = None
                    raw_bytes = None

                if img_b64:
                    try:
                        pil_img = Image.open(io.BytesIO(base64.b64decode(img_b64)))
                    except Exception:
                        pil_img = None
                elif raw_bytes:
                    try:
                        pil_img = Image.open(io.BytesIO(raw_bytes))
                    except Exception:
                        pil_img = None
                else:
                    pil_img = None

                if isinstance(pil_img, Image.Image):
                    out_path = img_dir / f"slide_{idx}_img_{len(image_paths)+1}.jpg"
                    saved = save_image_bytes(pil_img, out_path)
                    image_paths.append(saved)

        if not title and text_chunks:
            title = safe_text(text_chunks[0].split("\n")[0])[:120]
        slide_text = "\n".join(text_chunks).strip()

        if image_paths:
            for p in image_paths:
                rows.append({
                    "slide_number": idx,
                    "slide_title": title,
                    "slide_content": slide_text,
                    "has_image": True,
                    "image_path": os.path.relpath(p, sub_out_dir),
                    "extracted_content_from_image": "",
                })
        else:
            rows.append({
                "slide_number": idx,
                "slide_title": title,
                "slide_content": slide_text,
                "has_image": False,
                "image_path": "",
                "extracted_content_from_image": "",
            })
    return rows

def _fallback_python_pptx(ppt_path: Path, sub_out_dir: Path, img_dir: Path):
    rows = []
    prs = Presentation(str(ppt_path))
    for i, slide in enumerate(prs.slides, start=1):
        # Title
        title = ""
        if slide.shapes.title and slide.shapes.title.has_text_frame:
            title = safe_text(slide.shapes.title.text)

        # Text
        text_chunks = []
        for shape in slide.shapes:
            if shape.has_text_frame and shape != slide.shapes.title:
                txt = safe_text(shape.text)
                if txt:
                    text_chunks.append(txt)

        # Images
        image_paths = []
        for shape in slide.shapes:
            try:
                if shape.shape_type == MSO_SHAPE_TYPE.PICTURE:
                    image = shape.image
                    img_bytes = image.blob
                    pil_img = Image.open(io.BytesIO(img_bytes))
                    out_path = img_dir / f"slide_{i}_img_{len(image_paths)+1}.jpg"
                    saved = save_image_bytes(pil_img, out_path)
                    image_paths.append(saved)
            except Exception:
                pass

        if not title and text_chunks:
            title = safe_text(text_chunks[0].split("\n")[0])[:120]
        slide_text = "\n".join(text_chunks).strip()

        if image_paths:
            for p in image_paths:
                rows.append({
                    "slide_number": i,
                    "slide_title": title,
                    "slide_content": slide_text,
                    "has_image": True,
                    "image_path": os.path.relpath(p, sub_out_dir),
                    "extracted_content_from_image": "",
                })
        else:
            rows.append({
                "slide_number": i,
                "slide_title": title,
                "slide_content": slide_text,
                "has_image": False,
                "image_path": "",
                "extracted_content_from_image": "",
            })
    return rows

def extract_ppt_with_dpk(ppt_path: Path, out_root: Path) -> Path:
    """
    Robust extractor for PPT/PPTX:
      1) Try Docling page objects
      2) Try Docling to_dict() structure
      3) Fallback to python-pptx
    Then OCRs any exported images and writes parquet with required schema.
    """
    submission_id = ppt_path.stem
    sub_out_dir = out_root / submission_id
    img_dir = sub_out_dir / "images"
    sub_out_dir.mkdir(parents=True, exist_ok=True)
    img_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    #Always call Docling (DPK) first
    converter = DocumentConverter()
    result = converter.convert(str(ppt_path))

    doc = getattr(result, "document", None)
    pages = getattr(doc, "pages", None)
    if pages and hasattr(pages, "__iter__"):
        try:
            first = next(iter(pages))
            if hasattr(first, "blocks"):
                slide_idx = 0
                for page in pages:
                    slide_idx += 1
                    rows.extend(_collect_from_docling_page_blocks(page, sub_out_dir, img_dir, slide_idx))
        except StopIteration:
            pass

    # Strategy 2: dict view
    if not rows:
        try:
            doc_dict = doc.to_dict() if doc and hasattr(doc, "to_dict") else None
            if isinstance(doc_dict, dict):
                rows = _collect_from_docling_dict(doc_dict, sub_out_dir, img_dir)
        except Exception:
            rows = []

    # Strategy 3: python-pptx fallback
    if not rows:
        rows = _fallback_python_pptx(ppt_path, sub_out_dir, img_dir)

    # OCR pass
    for r in rows:
        if r["has_image"] and r["image_path"]:
            img_abs = sub_out_dir / r["image_path"]
            r["extracted_content_from_image"] = ocr_image_file(img_abs)

    # Write parquet
    df = pd.DataFrame(rows, columns=[
        "slide_number",
        "slide_title",
        "slide_content",
        "has_image",
        "image_path",
        "extracted_content_from_image",
    ])
    parquet_path = sub_out_dir / f"{submission_id}.parquet"
    df.to_parquet(parquet_path, index=False)
    return parquet_path

# ---------------- PDF → Parquet (page == slide) ----------------
def extract_pdf_to_parquet(pdf_path: Path, out_root: Path) -> Path:
    """
    Treat each PDF page like a slide; extract vector text, export images,
    OCR images and (if needed) rasterized full page when no text.
    Schema matches PPT extractor so downstream pipeline stays unchanged.
    """
    submission_id = pdf_path.stem
    sub_out_dir = out_root / submission_id
    img_dir = sub_out_dir / "images"
    sub_out_dir.mkdir(parents=True, exist_ok=True)
    img_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    doc = fitz.open(str(pdf_path))

    def _derive_title(page_text: str, page_index: int) -> str:
        for line in (page_text or "").splitlines():
            line = line.strip()
            if len(line) >= 3:
                return line[:120]
        return f"Page {page_index+1}"

    for i, page in enumerate(doc):
        txt = page.get_text("text") or ""
        page_has_img = False
        page_img_paths = []
        ocr_concat = []

        # Export embedded images and OCR them
        for img_idx, img_info in enumerate(page.get_images(full=True)):
            xref = img_info[0]
            try:
                pix = fitz.Pixmap(doc, xref)
                if pix.n > 4:
                    pix = fitz.Pixmap(fitz.csRGB, pix)
                out_img = img_dir / f"{submission_id}_p{i+1}_{img_idx+1}.png"
                pix.save(str(out_img))
                page_img_paths.append(os.path.relpath(out_img, sub_out_dir))
                page_has_img = True
                t = ocr_image_file(out_img)
                if isinstance(t, str) and t.strip():
                    ocr_concat.append(t.strip())
            except Exception:
                continue

        # If no vector text, rasterize the page and OCR
        extracted_from_page_image = ""
        if not txt.strip():
            try:
                pix = page.get_pixmap(dpi=220)
                out_full = img_dir / f"{submission_id}_p{i+1}_full.png"
                pix.save(str(out_full))
                page_img_paths.append(os.path.relpath(out_full, sub_out_dir))
                page_has_img = True
                extracted_from_page_image = (ocr_image_file(out_full) or "").strip()
            except Exception:
                extracted_from_page_image = ""

        if ocr_concat:
            if extracted_from_page_image:
                extracted_from_page_image = (extracted_from_page_image + "\n\n" + "\n\n".join(ocr_concat)).strip()
            else:
                extracted_from_page_image = "\n\n".join(ocr_concat)

        title = _derive_title(txt, i)
        rows.append({
            "submission_id": submission_id,
            "slide_number": i + 1,
            "slide_title": title,
            "slide_content": txt.strip(),
            "has_image": bool(page_has_img),
            "image_path": ";".join(page_img_paths),
            "extracted_content_from_image": extracted_from_page_image,
        })

    df = pd.DataFrame(rows, columns=[
        "submission_id",
        "slide_number",
        "slide_title",
        "slide_content",
        "has_image",
        "image_path",
        "extracted_content_from_image",
    ])
    parquet_path = sub_out_dir / f"{submission_id}.parquet"
    df.to_parquet(parquet_path, index=False)
    return parquet_path

In [ ]:
# @title Run extraction on all PPT/PPTX/PDF in input_submissions
from glob import glob
import traceback

def run_batch_extraction(input_dir: Path = INPUT_DIR, out_dir: Path = PIPELINE_OUT) -> pd.DataFrame:
    # Collect files
    ppt_files = []
    ppt_files += glob(str(input_dir / "*.pptx"))
    ppt_files += glob(str(input_dir / "*.ppt"))
    pdf_files = glob(str(input_dir / "*.pdf"))

    ppt_files = [Path(p) for p in ppt_files]
    pdf_files = [Path(p) for p in pdf_files]

    if not ppt_files and not pdf_files:
        print(f"No PPT/PPTX/PDF found in {input_dir}. Put some files there and re-run.")
        return pd.DataFrame()

    all_parquets = []
    third_slide_results = []  # To store results from third slide analysis (PPT only)

    # ---- Process PDFs first (page == slide) ----
    for pdf in pdf_files:
        try:
            print(f"Extracting PDF -> {pdf.name}")
            pq = extract_pdf_to_parquet(pdf, out_root=out_dir)
            all_parquets.append(pq)
            print(f"Wrote: {pq}")

            # Mark third-slide analysis as Not Applicable for PDFs
            third_slide_results.append({
                "submission_id": pdf.stem,
                "technologies_used": "",
                "impact_focus": "",
                "third_slide_text_ocr": "",
                "third_slide_analysis_status": "Not Applicable (PDF)"
            })
        except Exception as e:
            print(f"Failed for {pdf.name}: {e}")
            traceback.print_exc()

    # ---- Process PPT/PPTX with DPK ----
    for ppt in ppt_files:
        try:
            print(f"Extracting with DPK -> {ppt.name}")
            pq = extract_ppt_with_dpk(ppt, out_root=out_dir)
            all_parquets.append(pq)
            print(f"Wrote: {pq}")

            # Process the third slide (PPT only)
            slide_number = 3
            slide_image = extract_slide_image_from_ppt(str(ppt), slide_number)
            analysis_results = analyze_slide_image(slide_image)

            third_slide_results.append({
                "submission_id": ppt.stem,
                "technologies_used": ", ".join(analysis_results["technologies"]),
                "impact_focus": analysis_results["impact"],
                "third_slide_text_ocr": analysis_results["extracted_text"],
                "third_slide_analysis_status": "Success" if slide_image is not None and analysis_results["extracted_text"] else "No Image/Text Found"
            })

        except Exception as e:
            print(f"Failed for {ppt.name}: {e}")
            traceback.print_exc()

    # Create a DataFrame from third slide results
    third_slide_df = pd.DataFrame(third_slide_results)

    # Merge third-slide results into each parquet (PPT gets values; PDF gets N/A row)
    for pq_path in all_parquets:
        try:
            df = pd.read_parquet(pq_path)
            submission_id = Path(pq_path).stem
            third_slide_row = third_slide_df[third_slide_df['submission_id'] == submission_id]

            if not third_slide_row.empty:
                for col in third_slide_row.columns:
                    if col != 'submission_id':
                        df[col] = third_slide_row[col].iloc[0]
                df.to_parquet(pq_path, index=False)
                print(f"Merged third slide analysis into {pq_path}")
        except Exception as e:
            print(f"Error merging third slide analysis into {pq_path}: {e}")
            traceback.print_exc()

    # Summarize a few rows for the notebook output
    summaries = []
    for pq in all_parquets:
        try:
            df = pd.read_parquet(pq)
            k = min(3, len(df))
            summary_row = {
                "submission": Path(pq).stem,
                "rows": len(df),
                "images": int(df["has_image"].sum()),
                "preview_titles": "; ".join(df["slide_title"].head(k).tolist())
            }
            if "technologies_used" in df.columns:
                summary_row["third_slide_tech"] = df["technologies_used"].iloc[0]
                summary_row["third_slide_impact"] = df["impact_focus"].iloc[0]
            summaries.append(summary_row)
        except Exception as e:
            print(f"Error summarizing {pq}: {e}")
            summaries.append({"submission": Path(pq).stem, "error": "Summary failed"})

    return pd.DataFrame(summaries)

In [ ]:
# @title Certificate extraction details

import numpy as np
from difflib import SequenceMatcher
from pathlib import Path
import pandas as pd
import re # Added missing import for 're'

def load_parquet(pq_path: Path) -> pd.DataFrame:
    return pd.read_parquet(pq_path)

def save_parquet(df: pd.DataFrame, pq_path: Path):
    df.to_parquet(pq_path, index=False)

def load_datasheet(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    return df

COL_HINTS = {
    "title":  ["slide_title", "title"],
    "text":   ["extracted_content_from_image", "slide_content", "content"],
}

def find_col(df: pd.DataFrame, candidates) -> str | None:
    lowmap = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lowmap:
            return lowmap[c.lower()]
    return None

def normalize_space(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def canonical(s: str) -> str:
    s = normalize_space(s).lower()
    s = re.sub(r"[^a-z0-9 ]+", "", s)
    return re.sub(r"\s+", " ", s).strip()

def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, canonical(a), canonical(b)).ratio()

#certificate template rules
# We allow NAME / PROGRAM / CODE to vary. The scaffolding must exist in this order.
ANCHOR_1 = r"this certificate is presented to"
ANCHOR_2 = r"for the completion of"
ANCHOR_3 = r"\([A-Za-z]{2,}[-:\s]*[A-Za-z0-9]{4,}\)"
ANCHOR_4 = r"as indicated by this learner"

# Extract fields from text following your layout
NAME_PATTERNS = [
    r"presented to\s*\n?\s*(?P<name>[A-Za-z][A-Za-z .'-]{2,})"
]
CODE_PATTERNS = [
    r"\((?P<prefix>[A-Za-z]{2,})[-:\s]*(?P<code>[A-Za-z0-9]{4,})\)"
]
PLAN_ID_PATTERNS = [
    r"\((?P<plan_id>PLAN-[A-Z0-9]+)\)"
]
DATE_PATTERNS = [
    r"Completion date:\s*(?P<completion_date>[0-9]{1,2}\s+[A-Za-z]{3}\s+[0-9]{4}\s*\(GMT\))"
]

def extract_fields_from_text(raw_text: str) -> dict:
    text = raw_text or ""
    name, code, program, plan_id, completion_date = "", "", "", "", "" # Initialize new fields

    # NAME (after "presented to")
    for pat in NAME_PATTERNS:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            name = normalize_space(m.groupdict().get("name", ""))
            break

    # CODE: any "(LETTERS-ALPHANUM...)" form
    for pat in CODE_PATTERNS:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            prefix = normalize_space(m.groupdict().get("prefix", ""))
            tail   = normalize_space(m.groupdict().get("code", ""))
            if prefix or tail:
                code = f"{prefix}-{tail}".strip("-")
            break

    # Try to extract specific PLAN ID if present, otherwise rely on general CODE
    for pat in PLAN_ID_PATTERNS:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            plan_id = normalize_space(m.groupdict().get("plan_id", ""))
            if not code and plan_id: # If general code wasn't found, use plan_id as code
                code = plan_id
            break

    # PROGRAM:
    # Everything after "for the completion of" up to the generic code parentheses or the "As indicated..." anchor.
    # Use a non-greedy match with a lookahead to stop at either boundary.
    prog_stop = r"\([A-Za-z]{2,}[-:\s]*[A-Za-z0-9]{4,}\\|As indicated by this learner"
    m = re.search(r"for the completion of\s*(?P<p>.*?)(?=(" + prog_stop + r"))",
                  text, flags=re.IGNORECASE | re.DOTALL)
    if m:
        program = normalize_space(m.group("p"))
    else:
        # Fallback: old behavior (until end or As indicated...)
        m2 = re.search(r"for the completion of\s*(?P<p>.+)", text, flags=re.IGNORECASE | re.DOTALL)
        if m2:
            rem = text[m2.start("p"):].strip()
            rem = re.split(prog_stop, rem, flags=re.IGNORECASE)[0]
            program = normalize_space(rem)

    # COMPLETION DATE
    for pat in DATE_PATTERNS:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            completion_date = normalize_space(m.groupdict().get("completion_date", ""))
            break

    return {
        "name": name,
        "program": program,
        "code": code,
        "plan_id": plan_id,
        "completion_date": completion_date
    }

def template_score(text: str) -> float:
    """Score (0..1) based on presence + correct order of anchors."""
    t = text.lower()
    p1 = re.search(ANCHOR_1, t, flags=re.IGNORECASE)
    p2 = re.search(ANCHOR_2, t, flags=re.IGNORECASE)
    p3 = re.search(ANCHOR_3, t, flags=re.IGNORECASE)  # <-- now generic, not URL-locked
    p4 = re.search(ANCHOR_4, t, flags=re.IGNORECASE)

    present = [bool(p1), bool(p2), bool(p3), bool(p4)]
    base = sum(present) / 4.0  # 0..1

    # order bonus
    order_bonus = 0.0
    ps = [p1.start() if p1 else -1, p2.start() if p2 else -1, p3.start() if p3 else -1, p4.start() if p4 else -1]
    if all(p >= 0 for p in ps) and ps == sorted(ps):
        order_bonus = 0.2
    elif sum(present) >= 3:
        seq = [p for p in ps if p >= 0]
        if seq == sorted(seq):
            order_bonus = 0.1

    return min(1.0, base + order_bonus)

# ---- match vs roster ----
def pick_cols(ds: pd.DataFrame):
    cmap = {c.lower().strip(): c for c in ds.columns}
    # try a few common headers
    name_col =  cmap.get("name of the participant") or cmap.get("participant") or cmap.get("name") or next((c for c in ds.columns if "name" in c.lower()), None)
    prog_col =  cmap.get("name of the program") or cmap.get("program name") or cmap.get("course name") or next((c for c in ds.columns if "program" in c.lower() or "course" in c.lower()), None)
    code_col =  cmap.get("code of the program") or cmap.get("program code") or cmap.get("url code") or cmap.get("code") or next((c for c in ds.columns if "code" in c.lower()), None)
    if not (name_col and prog_col and code_col):
        raise ValueError("Datasheet must have columns for name / program / code (case-insensitive).")
    return name_col, prog_col, code_col

def best_roster_match(extracted: dict, ds: pd.DataFrame) -> tuple[float, dict]:
    name_col, prog_col, code_col = pick_cols(ds)

    best_score = -1.0
    best_row   = {}
    for _, r in ds.iterrows():
        s1 = similarity(extracted.get("name"," "),   str(r[name_col]))
        s2 = similarity(extracted.get("program"," "),str(r[prog_col]))
        s3 = similarity(extracted.get("code"," "),   str(r[code_col]))
        score = (s1 + s2 + s3) / 3.0
        if score > best_score:
            best_score, best_row = score, r.to_dict()
    return best_score, best_row

# ---- final certificate score ----
# Weighted combo: template (40%) + roster match (60%), then scale to 0..100
def compute_certificate_score(text: str, ds: pd.DataFrame) -> tuple[float, dict, dict]:
    csv_output_path = Path(csv_output_path)
    fields = extract_fields_from_text(text)
    t_score = template_score(text)                       # 0..1
    r_score, match_row = best_roster_match(fields, ds)   # 0..1-ish from difflib
    final = (0.4 * t_score + 0.6 * r_score) * 100.0
    return final, fields, match_row

In [ ]:
# @title Add score to the parquet
from tqdm import tqdm

def is_certificate_row(title_val: str, text_val: str) -> bool:
    t = (title_val or "") + "\n" + (text_val or "")
    t = t.lower()
    return "certificate" in t  # simple, reliable for your format

def update_parquet_with_certificate_score(pq_path: Path, roster_df: pd.DataFrame) -> dict:
    df = load_parquet(pq_path)

    # choose columns
    title_col = find_col(df, COL_HINTS["title"])
    text_col  = find_col(df, COL_HINTS["text"])

    if text_col is None:
        # no text at all -> add zeros and exit
        if "certificate_score" not in df.columns:
            df["certificate_score"] = 0.0
        save_parquet(df, pq_path)
        return {"parquet": str(pq_path), "updated_rows": 0, "max_score": 0.0}

    # ensure column exists
    if "certificate_score" not in df.columns:
        df["certificate_score"] = 0.0

    updated = 0
    max_score = 0.0
    extracted_samples = []

    for idx, row in df.iterrows():
        title = row.get(title_col, "") if title_col else ""
        text  = str(row.get(text_col, "") or "")

        if not is_certificate_row(title, text):
            continue

        score, fields, match = compute_certificate_score(text, roster_df)

        score = round(float(score), 2)
        df.at[idx, "certificate_score"] = score

        updated += 1
        max_score = max(max_score, score)

        extracted_samples.append({
            "row": int(idx),
            "score": score,
            **fields
        })

    # 🚫 DO NOT overwrite certificate_score with any global variable
    save_parquet(df, pq_path)

    return {
        "parquet": str(pq_path),
        "updated_rows": updated,
        "max_score": round(max_score, 2),
        "samples": extracted_samples[:3],
    }

def process_all_parquets(parquet_root: Path, data_csv: Path, results_csv: Path | None = None) -> pd.DataFrame:
    """
    Batch certificate scoring:
    - Loads roster CSV
    - Finds all .parquet under parquet_root recursively
    - Updates each parquet with certificate score (via update_parquet_with_certificate_score)
    - Ensures certificate_score column exists everywhere (0.0 if missing)
    - Optionally writes a summary CSV
    - Returns a dataframe of per-parquet logs
    """
    roster = load_datasheet(data_csv)

    parquet_root = Path(parquet_root)
    data_csv = Path(data_csv)

    pq_list = [Path(p) for p in glob.glob(str(parquet_root / "**/*.parquet"), recursive=True)]
    if not pq_list:
        print(f"No parquet files found under {parquet_root}")
        return pd.DataFrame()

    logs = []
    # tqdm might not exist in some envs; fallback gracefully
    try:
        iterator = tqdm(pq_list, desc="Updating parquets")
    except Exception:
        iterator = pq_list

    for pq in iterator:
        try:
            info = update_parquet_with_certificate_score(pq, roster)
            # standardize minimal fields
            if isinstance(info, dict):
                info.setdefault("parquet", str(pq))
            else:
                info = {"parquet": str(pq), "info": str(info)}
            logs.append(info)
        except Exception as e:
            logs.append({"parquet": str(pq), "error": str(e)})

    # ✅ FIX: ensure certificate_score exists using pq_list (NOT parquet_paths)
    for pq in pq_list:
        try:
            df = pd.read_parquet(pq)
            if "certificate_score" not in df.columns:
                df = df.copy()
                df["certificate_score"] = 0.0
                df.to_parquet(pq, index=False)
        except Exception as e:
            logs.append({"parquet": str(pq), "error": f"ensure_certificate_score_failed: {e}"})

    # Optional: write a compact CSV summary for reference
    if results_csv:
        try:
            results_csv = Path(results_csv)
            results_csv.parent.mkdir(parents=True, exist_ok=True)
            rows = []
            for l in logs:
                if "error" in l:
                    rows.append({
                        "parquet": l.get("parquet", ""),
                        "updated_rows": 0,
                        "max_score": 0.0,
                        "error": l.get("error", ""),
                    })
                else:
                    rows.append({
                        "parquet": l.get("parquet", ""),
                        "updated_rows": l.get("updated_rows", 0),
                        "max_score": l.get("max_score", 0.0),
                        "error": "",
                    })
            pd.DataFrame(rows).to_csv(results_csv, index=False)
        except Exception as e:
            logs.append({"parquet": "(summary)", "error": f"summary_csv_write_failed: {e}"})

    return pd.json_normalize(logs, sep=".")

In [ ]:
# @title Missing requirements checker (Problem/Solution/Architecture)

import os, re, glob, pathlib
from pathlib import Path
import pandas as pd

# Reuse your existing helpers from the eval cell:
REQUIRED_SECTIONS = ["problem_statement", "proposed_solution", "technical_architecture"]

def _has_required_sections(df: pd.DataFrame) -> dict:
    title_col   = find_col(df, COL["title"])   or "slide_title"
    content_col = find_col(df, COL["content"]) or "slide_content"

    # Normalize patterns once
    comp_pats = {
        sec: [re.compile(p, re.IGNORECASE) for p in TITLE_PATTERNS.get(sec, [])]
        for sec in REQUIRED_SECTIONS
    }

    present = {sec: False for sec in REQUIRED_SECTIONS}

    for _, row in df.iterrows():
        title = str(row.get(title_col, "") or "")
        body  = str(row.get(content_col, "") or "")
        blob  = f"{title}\n{body}"

        # First try your classifier by title
        sec = title_to_section(title)
        if sec in present:
            present[sec] = True

        # Also try content/title pattern hits (more forgiving)
        for sec2 in REQUIRED_SECTIONS:
            if present[sec2]:
                continue
            for pat in comp_pats[sec2]:
                if pat.search(blob):
                    present[sec2] = True
                    break

        # Early exit if all found
        if all(present.values()):
            break

    return present

def _submission_id_for(df: pd.DataFrame, pq_path: Path) -> str:
    sub_col = find_col(df, COL["submission_id"])
    if sub_col:
        val = df[sub_col].iloc[0]
        if isinstance(val, str) and val.strip():
            return val.strip()
    return pq_path.stem

def scan_missing_requirements(out_root: Path) -> tuple[pd.DataFrame, Path]:
    REQUIRED_SECTIONS = {
        "problem_statement": ["problem statement", "challenge", "pain point"],
        "proposed_solution": ["proposed solution", "solution", "approach"],
        "technical_architecture": ["technical architecture", "architecture", "system design", "tech stack"],
    }

    pq_list = []
    for root, _, files in os.walk(str(out_root)):
        for fn in files:
            if fn.lower().endswith(".parquet"):
                pq_list.append(Path(root) / fn)

    miss_rows = []
    for pq in pq_list:
        try:
            dfp = pd.read_parquet(pq)
        except Exception as e:
            miss_rows.append({
                "submission_id": pq.stem,
                "parquet_path": str(pq),
                "missing": "ALL",
                "error": f"open_failed: {e}"
            })
            continue

        title_col = next((c for c in ["slide_title", "title"] if c in dfp.columns), None)
        content_col = next((c for c in ["extracted_content_from_image", "slide_content", "content"] if c in dfp.columns), None)
        if content_col is None:
            miss_rows.append({
                "submission_id": pq.stem,
                "parquet_path": str(pq),
                "missing": "ALL",
                "error": "no_text_columns"
            })
            continue

        found = {k: False for k in REQUIRED_SECTIONS}
        for _, r in dfp.iterrows():
            t = (str(r.get(title_col, "")) if title_col else "")
            v = str(r.get(content_col, ""))
            text = f"{t}\n{v}".lower()
            for req, pats in REQUIRED_SECTIONS.items():
                if any(p in text for p in pats):
                    found[req] = True

        missing = [k for k, ok in found.items() if not ok]
        if missing:
            miss_rows.append({
                "submission_id": pq.stem,
                "parquet_path": str(pq),
                "missing": ", ".join(missing),
                "error": ""
            })

    scores_dir = Path(out_root) / "scores"
    scores_dir.mkdir(parents=True, exist_ok=True)
    csv_path = scores_dir / "missing_requirements.csv"

    miss_df = pd.DataFrame(miss_rows)
    if not miss_df.empty:
        miss_df.to_csv(csv_path, index=False)

    # IMPORTANT: always (DataFrame, Path) in this order
    return miss_df, csv_path


def _df_to_rows(df: pd.DataFrame):
    """Gradio Dataframe happily accepts a DataFrame; keep this shim for your current outputs."""
    return df

In [ ]:
# @title Sending things to the API

# --- Colab installs ---
!pip -q install ibm-watsonx-ai gradio pandas pyarrow tenacity python-dotenv

import os, re, json, glob, math, textwrap, pathlib
from typing import Dict, List, Any, Optional, Tuple
import pandas as pd
from dataclasses import dataclass
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

import gradio as gr

# Watsonx SDK
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models import ModelInference

DEFAULT_MODELS = [
    "ibm/granite-3-2-8b-instruct",
]

DEFAULT_DIM_ROWS = {
    "uniqueness": 0.25,
    "impact": 0.30,
    "completeness": 0.30,
    "ethics": 0.05,
    "presentation": 0.10,
}

DEFAULT_SEC_ROWS = {
    "problem_statement": 0.30,
    "proposed_solution": 0.35,
    "technical_architecture": 0.35,
}

DEFAULT_CERTIFICATE_WEIGHT = 0.10

GEN_PARAMS = {
    GenParams.MAX_NEW_TOKENS: 768,
    GenParams.TEMPERATURE: 0.2,
    GenParams.TOP_P: 0.9,
    GenParams.REPETITION_PENALTY: 1.1,
}

COL = {
    "slide_no":  ["slide_no", "slide_number", "slide index"],
    "title":     ["slide_title", "title", "slide name"],
    "content":   ["slide_content", "content", "text"],
    "has_image": ["has_image", "contains_image", "images_present"],
    "img_text":  ["extracted_content_from_image", "extracted_images", "image_extracted", "ocr_text"],
    "cert":      ["certificate_score", "certificate_validity", "certificate_similarity"],
    "submission_id": ["submission_id", "file_stem", "name"],
}

TARGET_SECTIONS = ["problem_statement", "proposed_solution", "technical_architecture"]

TITLE_PATTERNS = {
    "problem_statement":        [r"\bproblem\b", r"\bchallenge\b", r"\bpain\s*point\b"],
    "proposed_solution":        [r"\bsolution\b", r"\bproposal\b", r"\bapproach\b"],
    "technical_architecture":   [r"\barchitecture\b", r"\bblueprint\b", r"\bsystem\s*design\b", r"\btech\s*stack\b", r"\barchitecture\s*blueprint\b"],
}

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def find_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None

def load_parquet_or_csv(path: str) -> pd.DataFrame:
    if path.lower().endswith(".csv"):
        return pd.read_csv(path)
    return pd.read_parquet(path)

#Missing requirements checker
REQUIRED_SECTIONS = {
    "problem_statement": ["problem statement", "challenge", "pain point"],
    "proposed_solution": ["proposed solution", "solution", "approach"],
    "technical_architecture": ["technical architecture", "architecture", "system design", "tech stack"]
}

def scan_missing_requirements(parquet_path: pathlib.Path) -> dict:
    """
    Check if required sections are present in slide titles/contents.
    Returns dict like: {"submission_id": ..., "missing": [..]}.
    """
    try:
        df = pd.read_parquet(parquet_path)
    except Exception as e:
        return {"submission_id": parquet_path.stem, "error": f"Could not open: {e}"}

    title_col = find_col(df, COL.get("title", [])) or "slide_title"
    content_col = find_col(df, COL.get("content", [])) or "slide_content"

    found = {k: False for k in REQUIRED_SECTIONS}
    for _, row in df.iterrows():
        text = str(row.get(title_col, "")).lower() + " " + str(row.get(content_col, "")).lower()
        for req, patterns in REQUIRED_SECTIONS.items():
            if any(pat in text for pat in patterns):
                found[req] = True

    missing = [req for req, ok in found.items() if not ok]
    return {"submission_id": parquet_path.stem, "missing": missing}

def normalize_weights(d: Dict[str, float]) -> Dict[str, float]:
    d = {k: float(v) for k,v in d.items()}
    s = sum(max(0.0, v) for v in d.values())
    if s == 0:
        # if all zeros, fallback to uniform
        n = len(d) or 1
        return {k: 1.0/n for k in d}
    return {k: max(0.0, v)/s for k, v in d.items()}

def ensure_relevance_dimension(dimensions: Dict[str, float], theme: Optional[Dict[str, Any]]) -> Dict[str, float]:
    """
    If a theme is provided but 'relevance_to_theme' isn't in the user's dimensions,
    add it with a reasonable default weight so it participates in scoring.
    """
    dims = dict(dimensions)
    if theme and "relevance_to_theme" not in dims:
        # default 0.15 is sensible; will be normalized anyway
        dims["relevance_to_theme"] = 0.15
    return dims

def title_to_section(title: str) -> Optional[str]:
    t = (title or "").strip().lower()
    for sec, pats in TITLE_PATTERNS.items():
        for p in pats:
            if re.search(p, t, flags=re.IGNORECASE):
                return sec
    return None

def chunk_text(text: str, max_chars=3000, overlap=250) -> List[str]:
    text = text.strip()
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]
    chunks, i = [], 0
    while i < len(text):
        chunk = text[i:i+max_chars]
        chunks.append(chunk)
        i += max_chars - overlap
    return chunks

@dataclass
class EvalConfig:
    dimensions: Dict[str, float]           # (normalized) dim -> weight
    section_weights: Dict[str, float]      # (normalized) section -> weight
    certificate_weight: float              # in [0,1]
    models: List[str]
    best_of_mode: str = "max"              # "max" | "capped_max"
    cap_above_median_pct: float = 0.10     # for capped_max

def build_prompt(section_name: str, section_text: str, dim_keys: List[str], theme: Optional[Dict[str, Any]] = None) -> str:
    dims_block = ",\n    ".join([f"\"{k}\": <float 0-10>" for k in dim_keys])
    theme_block = ""
    if theme:
        name = theme.get("name", "Custom Theme")
        desc = theme.get("description", "")
        keywords = theme.get("keywords", [])
        must_haves = theme.get("must_haves", [])
        forbidden = theme.get("forbidden", [])
        theme_block = f"""
THEME CONTEXT:
- Name: {name}
- Description: {desc}
- Keywords: {', '.join(keywords) if isinstance(keywords, list) else keywords}
- Must-haves (evidence expected in content): {', '.join(must_haves) if isinstance(must_haves, list) else must_haves}
- Forbidden (should penalize relevance): {', '.join(forbidden) if isinstance(forbidden, list) else forbidden}

When scoring "relevance_to_theme" (if present in keys):
- Give higher scores if content strongly matches theme name/description/keywords AND cites must-haves.
- Penalize if forbidden items appear or content is off-topic.
- 0 = completely off-topic, 10 = perfectly aligned and clearly evidenced.
"""

    return f"""You are a strict hackathon evaluator.
Rate the SECTION_CONTENT for "{section_name}" and RETURN ONLY JSON (no prose).
Scores must be numeric 0–10 for each dimension key exactly as listed.

{theme_block.strip()}

Respond as STRICT JSON:
{{
  "section": "{section_name}",
  "scores": {{
    {dims_block}
  }},
  "notes": "<1-2 line justification>"
}}

SECTION_CONTENT:
\"\"\"{section_text.strip()}\"\"\""""

class LLMReplyError(Exception): pass

def env_client(url: str, api_key: str) -> APIClient:
    if not url or not api_key:
        raise RuntimeError("Missing WATSONX_URL or WATSONX_API_KEY.")
    return APIClient(credentials={"url": url, "apikey": api_key})

def get_inference(client: APIClient, model_id: str, project_id: str) -> ModelInference:
    if not project_id:
        raise RuntimeError("Missing WATSONX_PROJECT_ID.")
    return ModelInference(model_id=model_id, params=GEN_PARAMS, api_client=client, project_id=project_id)

def _extract_json_block(raw: str) -> str:
    raw = (raw or "").strip()
    s, e = raw.find("{"), raw.rfind("}")
    if s != -1 and e != -1 and e > s:
        return raw[s:e+1]
    # fallback: try last {...}
    m = re.findall(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', raw, re.DOTALL)
    if m:
        return m[-1]
    raise LLMReplyError("No JSON object found in model response")

def _parse_reply(json_text: str, dim_keys: List[str]) -> Tuple[Dict[str, float], str]:
    try:
        data = json.loads(json_text)
    except Exception:
        jt = json_text.replace("'", '"').replace("None", "null").replace("True","true").replace("False","false")
        data = json.loads(jt)
    scores = data.get("scores", {})
    out = {}
    for k in dim_keys:
        try:
            out[k] = float(scores.get(k, 0.0))
        except Exception:
            out[k] = 0.0
        out[k] = max(0.0, min(10.0, out[k]))
    notes = str(data.get("notes", "") or "").strip()
    # keep it punchy
    if len(notes) > 240:
        notes = notes[:237].rstrip() + "..."
    return out, notes

@retry(reraise=True, stop=stop_after_attempt(3), wait=wait_exponential(min=1, max=8),
       retry=retry_if_exception_type((LLMReplyError, TimeoutError)))
def score_text(mi: ModelInference, section_name: str, section_text: str, dim_keys: List[str], theme: Optional[Dict[str, Any]] = None) -> Tuple[Dict[str, float], str]:
    prompt = build_prompt(section_name, section_text, dim_keys, theme=theme)
    resp = mi.generate_text(prompt=prompt, params=GEN_PARAMS, raw_response=True)
    raw = resp["results"][0]["generated_text"] if isinstance(resp, dict) else str(resp)
    jb = _extract_json_block(raw)
    scores, notes = _parse_reply(jb, dim_keys)
    return scores, notes

def section_total_0_100(dim_scores_0_10: Dict[str, float], dim_weights: Dict[str, float]) -> float:
    # 0–10 to 0–1, weight, then ×100
    total = sum((dim_scores_0_10[k] / 10.0) * dim_weights.get(k, 0.0) for k in dim_weights)
    return max(0.0, min(100.0, total * 100.0))

def build_section_texts(df: pd.DataFrame) -> Tuple[Dict[str, str], Dict[str, bool], Dict[str, str]]:
    """
    Returns:
      section_texts: section -> concatenated text
      used_image_text: section -> bool (if any row used extracted image text)
      source_titles: section -> sample title joined
    """
    title_col = find_col(df, COL["title"]) or "slide_title"
    content_col = find_col(df, COL["content"]) or "slide_content"
    has_image_col = find_col(df, COL["has_image"]) or "has_image"
    img_text_col = find_col(df, COL["img_text"]) or "extracted_content_from_image"

    rows = df.to_dict("records")

    buckets = {sec: [] for sec in TARGET_SECTIONS}
    used_img = {sec: False for sec in TARGET_SECTIONS}
    titles_for_sec = {sec: [] for sec in TARGET_SECTIONS}

    for r in rows:
        sec = title_to_section(str(r.get(title_col, "")))
        if sec not in TARGET_SECTIONS:
            continue

        has_img = str(r.get(has_image_col, "")).strip().lower() in ("true","1","yes")
        img_txt = (r.get(img_text_col) or "").strip()
        txt = (r.get(content_col) or "").strip()

        chosen_text = (img_txt if (has_img and img_txt) else txt)
        if not chosen_text:
            continue

        buckets[sec].append(chosen_text)
        if has_img and img_txt:
            used_img[sec] = True
        titles_for_sec[sec].append(str(r.get(title_col, "")).strip())

    section_texts = {}
    for sec in TARGET_SECTIONS:
        if buckets[sec]:
            joined = "\n\n".join(f"[Slide {i+1}]\n{t}" for i, t in enumerate(buckets[sec]))
            section_texts[sec] = joined
        else:
            section_texts[sec] = ""

    titles = {sec: " | ".join(titles_for_sec[sec][:3]) for sec in TARGET_SECTIONS}
    return section_texts, used_img, titles

def eval_submission(
    parquet_path: str,
    client: APIClient,
    model_ids: List[str],
    project_id: str,
    cfg: EvalConfig,
    log: List[str],
    theme: Optional[Dict[str, Any]] = None,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:

    df = load_parquet_or_csv(parquet_path)
    section_texts, used_image_text, source_titles = build_section_texts(df)

    sub_id = pathlib.Path(parquet_path).stem
    sub_col = find_col(df, COL["submission_id"])
    if sub_col and isinstance(df[sub_col].iloc[0], str):
        sub_id = df[sub_col].iloc[0]

    cert_score = 0.0
    cert_col = find_col(df, COL["cert"])
    if cert_col:
        try:
            s = pd.to_numeric(df[cert_col], errors="coerce").dropna()
            if len(s):
                cert_score = float(s.max())
        except Exception:
            cert_score = 0.0

    model_infers = {}
    for mid in model_ids:
        try:
            model_infers[mid] = get_inference(client, mid, project_id)
        except Exception as e:
            log.append(f"Model init failed for {mid}: {e}")

    dim_keys = list(cfg.dimensions.keys())
    per_model_rows: List[Dict[str, Any]] = []
    per_section_best: Dict[str, float] = {}
    per_section_best_feedback: Dict[str, str] = {}

    for sec in TARGET_SECTIONS:
        text = section_texts.get(sec, "").strip()
        if not text:
            per_section_best[sec] = 20.0
            per_section_best_feedback[sec] = "No content detected for this section."
            continue

        chunks = chunk_text(text, max_chars=3000, overlap=250)

        model_section_totals: Dict[str, float] = {}
        model_section_feedbacks: Dict[str, str] = {}

        for mid, mi in model_infers.items():
            dim_acc = {k: 0.0 for k in dim_keys}
            counted = 0
            first_note = ""
            for ch in chunks:
                try:
                    scores_0_10, note = score_text(mi, sec, ch, dim_keys, theme=theme)
                    if note and not first_note:
                        first_note = note
                    for k in dim_keys:
                        dim_acc[k] += scores_0_10[k]
                    counted += 1
                except Exception as e:
                    log.append(f"{sub_id}:{sec}:{mid} chunk scoring failed: {e}")

            if counted == 0:
                section_total = 50.0
                dim_avg = {k: 5.0 for k in dim_keys}
                section_note = first_note or "Model failed on chunks; assigned neutral score."
            else:
                dim_avg = {k: dim_acc[k]/counted for k in dim_keys}
                section_total = section_total_0_100(dim_avg, cfg.dimensions)
                section_note = first_note or "Scored successfully."

            model_section_totals[mid] = section_total
            model_section_feedbacks[mid] = section_note

            per_model_rows.append({
                "submission_id": sub_id,
                "model_id": mid,
                "section": sec,
                **{f"dim_{k}_0_10": dim_avg[k] for k in dim_keys},
                "section_total_0_100": section_total,
                "section_feedback": section_note,     # <— NEW
                "used_image_text": bool(used_image_text.get(sec, False)),
                "source_titles": source_titles.get(sec, "")
            })

        if model_section_totals:
            if cfg.best_of_mode == "max":
                best_mid = max(model_section_totals, key=model_section_totals.get)
                per_section_best[sec] = model_section_totals[best_mid]
                per_section_best_feedback[sec] = model_section_feedbacks.get(best_mid, "")
            else:
                vals = sorted(model_section_totals.values())
                med = vals[len(vals)//2] if vals else 0.0
                cap = med * (1.0 + cfg.cap_above_median_pct)
                # pick model that achieved the chosen capped score
                target = min(max(vals) if vals else 0.0, cap)
                # choose the model closest to target from above; fallback to best
                best_mid = max(model_section_totals, key=model_section_totals.get)
                chosen_mid = None
                for m, v in model_section_totals.items():
                    if abs(v - target) < 1e-6:
                        chosen_mid = m
                        break
                if not chosen_mid:
                    chosen_mid = best_mid
                per_section_best[sec] = model_section_totals[chosen_mid]
                per_section_best_feedback[sec] = model_section_feedbacks.get(chosen_mid, "")
        else:
            per_section_best[sec] = 50.0
            per_section_best_feedback[sec] = "No usable model output; assigned neutral score."

    # weighted by section weights
    base = sum(per_section_best[s] * cfg.section_weights.get(s, 0.0) for s in TARGET_SECTIONS)
    final = base * (1.0 - cfg.certificate_weight) + (cert_score * cfg.certificate_weight)
    final = max(0.0, min(100.0, final))

    overall_feedback = " | ".join([
        f"Problem: {per_section_best_feedback.get('problem_statement','')}".strip(),
        f"Solution: {per_section_best_feedback.get('proposed_solution','')}".strip(),
        f"Architecture: {per_section_best_feedback.get('technical_architecture','')}".strip(),
    ]).strip()

    final_row = {
        "submission_id": sub_id,
        **{f"best_{sec}": per_section_best[sec] for sec in TARGET_SECTIONS},
        "certificate_score": cert_score,
        "final_score": final,
        "feedback": overall_feedback,  # <— NEW
    }

    return per_model_rows, final_row

def run_evaluation(
    input_dir: str,
    watsonx_url: str,
    watsonx_api_key: str,
    watsonx_project_id: str,
    models_csv: str,
    dimensions_json: str,
    section_weights_json: str,
    certificate_weight: float,
    out_prefix: str = "scores/run",
    best_of_mode: str = "max",
    cap_pct: float = 0.10,
    theme: Optional[Dict[str, Any]] = None,
) -> Tuple[str, pd.DataFrame]:
    log: List[str] = []
    ensure_dir("scores")

    try:
        models = [m.strip() for m in (models_csv or "").split(",") if m.strip()]
        if not models:
            models = DEFAULT_MODELS
    except Exception:
        models = DEFAULT_MODELS

    try:
        dims = json.loads(dimensions_json) if dimensions_json else DEFAULT_DIM_ROWS
    except Exception as e:
        log.append(f"dimensions_json parse failed, using defaults: {e}")
        dims = DEFAULT_DIM_ROWS
    # always ensure theme relevance if theme provided
    dims = ensure_relevance_dimension(dims, theme)

    try:
        secw = json.loads(section_weights_json) if section_weights_json else DEFAULT_SEC_ROWS
    except Exception as e:
        log.append(f"section_weights_json parse failed, using defaults: {e}")
        secw = DEFAULT_SEC_ROWS

    dims = normalize_weights(dims)
    secw = normalize_weights(secw)
    certificate_weight = max(0.0, min(1.0, float(certificate_weight if certificate_weight is not None else DEFAULT_CERTIFICATE_WEIGHT)))

    cfg = EvalConfig(
        dimensions=dims,
        section_weights=secw,
        certificate_weight=certificate_weight,
        models=models,
        best_of_mode=best_of_mode,
        cap_above_median_pct=float(cap_pct)
    )

    try:
        client = env_client(watsonx_url, watsonx_api_key)
    except Exception as e:
        return f"Watsonx client init failed: {e}", pd.DataFrame()

    paths = []
    for root, _, files in os.walk(input_dir):
        for fn in files:
            if fn.lower().endswith(".parquet"):
                paths.append(os.path.join(root, fn))
    paths = sorted(paths)
    if not paths:
        return f"No .parquet files in {input_dir}", pd.DataFrame()

    per_model_records: List[Dict[str, Any]] = []
    final_records: List[Dict[str, Any]] = []

    for p in paths:
        log.append(f"Evaluating -> {os.path.basename(p)}")
        try:
            per_model_rows, final_row = eval_submission(
                parquet_path=p,
                client=client,
                model_ids=models,
                project_id=watsonx_project_id,
                cfg=cfg,
                log=log,
                theme=theme
            )
            per_model_records.extend(per_model_rows)
            final_records.append(final_row)
            log.append(f"Done -> {final_row['submission_id']}: final={final_row['final_score']:.2f}")
        except Exception as e:
            log.append(f"Failed -> {os.path.basename(p)}: {e}")

    per_model_df = pd.DataFrame(per_model_records)
    final_df = pd.DataFrame(final_records)

    per_model_path = "drive/MyDrive/hackathon_ppts/scores/per_model_score.parquet"
    final_path = "drive/MyDrive/hackathon_ppts/scores/final_scores.parquet"
    top20_path = "drive/MyDrive/hackathon_ppts/scores/top20.csv"

    if not per_model_df.empty:
        per_model_df.to_parquet(per_model_path, index=False)

    if not final_df.empty:
        final_df.sort_values("final_score", ascending=False, inplace=True)

        import math
        top_n = math.ceil(0.2 * len(final_df))
        top_scores = final_df['final_score'].nlargest(top_n)
        avg_top20 = top_scores.mean()

        if avg_top20 > 0:
            scaling_factor = 90 / avg_top20
        else:
            scaling_factor = 1.0

        final_df['final_score_scaled'] = final_df['final_score'] * scaling_factor
        final_df['final_score_scaled'] = final_df['final_score_scaled'].clip(upper=98)

        final_df.to_parquet(final_path, index=False)
        final_df.head(20).to_csv(top20_path, index=False)

    summary = [
        f"Processed: {len(paths)} files",
        f"Models: {', '.join(models)}",
        f"Dimensions: {cfg.dimensions}",
        f"Section weights: {cfg.section_weights}",
        f"Certificate weight: {cfg.certificate_weight}",
        f"Wrote: {per_model_path if not per_model_df.empty else '(no per-model rows)'}",
        f"Wrote: {final_path if not final_df.empty else '(no final rows)'}",
        f"Wrote: {top20_path if not final_df.empty else '(no final rows)'}",
    ]
    log_text = "\n".join(log + ["", *summary])

    top20 = final_df.head(20) if not final_df.empty else pd.DataFrame()
    return log_text, top20

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 34.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 15.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 150.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.


In [ ]:
# @title Categorisation Extraction
import os
import io
from pathlib import Path
import pandas as pd
from pptx import Presentation
from PIL import Image
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

def extract_slide_image_from_ppt(ppt_path, slide_number):
    try:
        prs = Presentation(ppt_path)
        if slide_number < 1 or slide_number > len(prs.slides):
            print(f"Warning: Slide number {slide_number} out of range for {ppt_path}. Has {len(prs.slides)} slides.")
            return None

        slide = prs.slides[slide_number - 1]
        max_img_size = 0
        slide_image = None

        for shape in slide.shapes:
            if hasattr(shape, 'image'):
                img = shape.image
                image_bytes = img.blob
                try:
                    pil_img = Image.open(io.BytesIO(image_bytes))
                    current_size = pil_img.width * pil_img.height
                    if current_size > max_img_size:
                        max_img_size = current_size
                        slide_image = pil_img
                except Exception as e:
                    print(f"Error opening image in {ppt_path}, slide {slide_number}: {e}")
                    continue

        return slide_image

    except Exception as e:
        print(f"Error extracting slide {slide_number} from {ppt_path}: {e}")
        return None

def analyze_slide_image_with_docling(pil_image):
    """
    Analyze a PIL image using Docling to extract text, technologies, and impact focus.
    """
    if pil_image is None:
        return {"technologies": [], "impact": None, "extracted_text": ""}

    try:
        # Convert PIL image to bytes
        img_byte_arr = io.BytesIO()
        pil_image.save(img_byte_arr, format='PNG')
        img_bytes = img_byte_arr.getvalue()

        # Run Docling OCR via DocumentConverter
        result = converter.convert_bytes(img_bytes, input_format="image/png")
        doc = result.document

        extracted_text = " ".join([block.text for page in doc.pages for block in page.blocks if hasattr(block, "text")])

        technologies = ["Data Prep Kit", "Granite", "RAG", "Agentic"]
        used_technologies = [t for t in technologies if t.lower() in extracted_text.lower()]

        impact = None
        if "Social Impact" in extracted_text:
            impact = "Social Impact"
        elif "Business Impact" in extracted_text:
            impact = "Business Impact"

        return {
            "technologies": used_technologies,
            "impact": impact,
            "extracted_text": extracted_text
        }

    except Exception as e:
        print(f"Docling analysis error: {e}")
        return {"technologies": [], "impact": None, "extracted_text": f"[ERROR] {e}"}

def process_third_slide_batch(ppt_file_paths, output_parquet_path):
    """
    Process the third slide of each PPT file and save results to a parquet file.
    Merges with existing data if output_parquet_path exists.
    """
    results = []
    for ppt_path_str in ppt_file_paths:
        ppt_path = Path(ppt_path_str)
        submission_id = ppt_path.stem
        slide_number = 3

        slide_image = extract_slide_image_from_ppt(str(ppt_path), slide_number)
        analysis_results = analyze_slide_image_with_docling(slide_image)

        results.append({
            "submission_id": submission_id,
            "slide_number": slide_number,
            "technologies_used": ", ".join(analysis_results["technologies"]),
            "impact_focus": analysis_results["impact"],
            "third_slide_text_ocr": analysis_results["extracted_text"],
            "third_slide_analysis_status": "Success" if slide_image is not None and analysis_results["extracted_text"] else "No Image/Text Found"
        })

    new_df = pd.DataFrame(results)

    # Merge with existing parquet if it exists
    if Path(output_parquet_path).exists():
        try:
            existing_df = pd.read_parquet(output_parquet_path)
            merged_df = pd.merge(existing_df, new_df, on="submission_id", how="left", suffixes=('', '_new'))

            # Drop redundant _new columns
            cols_to_drop = [col for col in merged_df.columns if col.endswith('_new') and col[:-4] in existing_df.columns]
            merged_df.drop(columns=cols_to_drop, inplace=True)

            # Ensure all submissions included
            all_submission_ids = pd.concat([existing_df['submission_id'], new_df['submission_id']]).unique()
            final_df = pd.DataFrame({'submission_id': all_submission_ids})
            final_df = pd.merge(final_df, existing_df, on='submission_id', how='left')
            final_df = pd.merge(final_df, new_df, on='submission_id', how='left', suffixes=('', '_new_analysis'))

            # Merge columns from new_df into final_df
            for col in new_df.columns:
                if col != 'submission_id':
                    if f'{col}_new_analysis' in final_df.columns:
                        final_df[col] = final_df[f'{col}_new_analysis'].fillna(final_df[col])
                        final_df.drop(columns=[f'{col}_new_analysis'], inplace=True)
                    elif col not in final_df.columns:
                        final_df[col] = final_df[col]

        except Exception as e:
            print(f"Error merging with existing parquet {output_parquet_path}: {e}. Saving new data only.")
            final_df = new_df
    else:
        final_df = new_df

    final_df.to_parquet(output_parquet_path, index=False)
    print(f"Processed {len(ppt_file_paths)} PPTs and saved results to {output_parquet_path}")
    return final_df


In [ ]:
# @title Path names, models, parameters for scoring

import os, glob, json, pathlib, textwrap, inspect, shutil, tempfile, re, threading, time, warnings
from pathlib import Path
from typing import Optional, Dict, Any
import pandas as pd
import gradio as gr

# Silence noisy but harmless PIL warning
warnings.filterwarnings(
    "ignore",
    message="Palette images with Transparency expressed in bytes should be converted to RGBA images",
)

EXTRACT_CSV_NAME       = "extraction_certificate_scores.csv"
EXTRACT_PARQUET_NAME   = "extraction_certificate_scores.parquet"
MASTER_SLIDES_PARQUET  = "master_slides.parquet"
EVAL_CSV_NAME          = "evaluation_results.csv"
EVAL_PARQUET_NAME      = "evaluation_results.parquet"
THEME_TRACK_PARQUET    = "theme_track_tech.parquet"

_AGG_FILES = {
    EXTRACT_PARQUET_NAME,
    EXTRACT_CSV_NAME,
    MASTER_SLIDES_PARQUET,
    EVAL_PARQUET_NAME,
    EVAL_CSV_NAME,
}

# Watsonx default models (unchanged; deprecation warnings may appear, but kept as requested)
WATSONX_DEFAULT_MODELS = [
    "openai/gpt-oss-120b",
    "ibm/granite-3-2-8b-instruct",
    "meta-llama/llama-3-3-70b-instruct",
    "ibm/granite-4-h-small",
]

# Hugging Face preset models when provider = "Hugging Face"
HF_PRESET_MODELS = [
    "mistralai/Mistral-7B-Instruct-v0.3",
    "ibm-granite/granite-3.3-8b-instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
]

# Default dimension & section weights
DEFAULT_DIM_ROWS = [
    ["Problem Statement", 0.10],
    ["Proposed Solution", 0.15],
    ["Technical Architecture", 0.15],
    ["Novelty  Uniqueness/creativity", 0.15],
    ["presentation", 0.10],
    ["ethical consideration", 0.10],
    ["completeness of design solution", 0.15],
    ["implementing ibm dpk/rag/aws/watsonx/granite", 0.10],
]
DEFAULT_SEC_ROWS = [
    ["problem_statement", 0.30],
    ["proposed_solution", 0.35],
    ["technical_architecture", 0.35],
]

# Detailed evaluation criterion columns
CRITERION_COLUMNS = [c[0] for c in DEFAULT_DIM_ROWS]

DETAILED_CRITERIA = {
    "Problem Statement": [
        "Is the problem statement clearly defined and realistic?",
        "Is the problem well-articulated and meaningful?"
    ],
    "Proposed Solution": [
        "Is the proposed solution comprehensive and well-designed?",
        "Does the solution effectively address the problem statement?"
    ],
    "Technical Architecture": [
        "Is the technical architecture clearly presented?",
        "Are system components and their interactions explained?",
        "Is the technology stack appropriate for the solution?",
        "Are technical diagrams present and clear?"
    ],
    "Novelty  Uniqueness/creativity": [
        "Is the technology suggested feasible to implement?",
        "Is there a working prototype?",
        "How have you tested your solution for any bias?",
        "What is unique/original/creative in this solution?"
    ],
    "presentation": [
        "Is the presentation clear, concise and structured?",
        "Is 'Why', 'What', and 'How' explained?",
        "Are services used clearly mentioned?",
        "Is the LLM used specified?",
        "Is training content used described?",
        "Is the architecture explained?"
    ],
    "ethical consideration": [
        "Is it ethical to implement the idea?",
        "Have you considered factors like gender neutrality?",
        "Are racial or religious biases being avoided?",
        "What bias testing methodology was used?",
        "How does the solution consider ethical implications?"
    ],
    "completeness of design solution": [
        "How is the solution making things better?",
        "What sort of deviation (positive/negative) can be seen by adopting your solution?",
        "Will it be scalable? Where else can this be used to make an impact?",
        "What can your solution do that just a simple Google search does not do?",
        "How would end-users use it?",
        "What is the potential positive impact on the chosen theme?"
    ],
    "implementing ibm dpk/rag/aws/watsonx/granite": [
        "Have you used one of the must-haves in your solution?",
        "What are the must-have techs that have been used in your solution?",
        "If using Traditional AI, is DPK usage present (mandatory)?",
        "How different is the solution compared to other LLMs being used?",
        "How is this solution better than using LLMs/Gen AI?",
        "Is IBM or AWS or both mentioned and used?",
        "Is RAG/Agentic usage clearly demonstrated?",
        "Are DPK/IBM Granite models used (RAG/Agentic Usage)?"
    ]
}

# Tick-based technology detection config
_TECH_TERMS = ["Data Prep Kit", "Granite", "RAG", "Agentic"]
TICK_CHARS = set(list("✓✔☑✅🗹🗸√"))
YES_PATTERN = re.compile(r"\byes\b", re.I)
NO_PATTERN = re.compile(r"\bno\b", re.I)

DEFAULT_DIM_JSON = json.dumps(DEFAULT_DIM_ROWS, indent=2)
DEFAULT_SECW_JSON = json.dumps(DEFAULT_SEC_ROWS, indent=2)

WX_URL_DEFAULT = os.environ.get("WATSONX_URL", "")
WX_KEY_DEFAULT = os.environ.get("WATSONX_API_KEY", "")
WX_PROJECT_DEFAULT = os.environ.get("WATSONX_PROJECT_ID", "")
HF_TOKEN_DEFAULT = os.environ.get("HUGGING_FACE_HUB_TOKEN", os.environ.get("HF_API_TOKEN", ""))

try:
    _HAS_CERT_FUNCS = callable(globals().get("load_datasheet")) and callable(globals().get("update_parquet_with_certificate_score"))
except Exception:
    _HAS_CERT_FUNCS = False

In [ ]:
# @title Tokens

def _apply_hf_token(token: str) -> bool:
    tok = (token or "").strip()
    if not tok:
        return False
    try:
        os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
        os.environ["HF_API_TOKEN"] = tok
        try:
            from huggingface_hub import set_access_token
            try:
                set_access_token(tok)
            except Exception:
                from huggingface_hub import HfFolder
                HfFolder.save_token(tok)
        except Exception:
            pass
        return True
    except Exception:
        return False

def _ensure_dir(p): os.makedirs(p, exist_ok=True)
def _list_parquets(dir_path: str | Path) -> list[str]:
    p = Path(dir_path)
    return sorted([str(x) for x in p.glob("*.parquet")])
def _parse_csv_line(s: str) -> list[str]:
    return [x.strip() for x in (s or "").split(",") if x.strip()]
def _build_theme_dict(name: str, desc: str, kw: str, must: str = "", forb: str = "") -> dict:
    theme = {"name": (name or "").strip() or "Custom Theme", "description": (desc or "").strip(),
             "keywords": _parse_csv_line(kw), "must_haves": _parse_csv_line(must), "forbidden": _parse_csv_line(forb)}
    if not any([theme["description"], theme["keywords"], theme["must_haves"], theme["forbidden"]]): return {}
    return theme
def _chunked(iterable, n):
    for i in range(0, len(iterable), n):
        yield iterable[i:i+n]
def _unique_parquet_path(out_dir: Path, base_stem: str) -> Path:
    candidate = out_dir / f"{base_stem}.parquet"
    if not candidate.exists(): return candidate
    k = 2
    while True:
        candidate = out_dir / f"{base_stem}_v{k}.parquet"
        if not candidate.exists(): return candidate
        k += 1

def _fallback_extract_single(ppt_path: str, out_dir: str):
    try:
        from pptx import Presentation
        prs = Presentation(ppt_path)
        rows = []
        for idx, slide in enumerate(prs.slides, start=1):
            title_text = ""
            for shape in slide.shapes:
                if hasattr(shape, "text_frame") and shape.text_frame and shape.text_frame.text:
                    title_text = (shape.text_frame.text or "").strip()
                    if title_text: break
            body_parts = []
            for shape in slide.shapes:
                if hasattr(shape, "text_frame") and shape.text_frame:
                    txt = (shape.text_frame.text or "").strip()
                    if txt: body_parts.append(txt)
            rows.append({"slide_no": idx, "slide_title": title_text, "slide_content": "\n".join(body_parts),
                         "has_image": False, "extracted_content_from_image": "", "image_path": ""})
        df = pd.DataFrame(rows)
        _ensure_dir(out_dir)
        out_path = os.path.join(out_dir, f"{pathlib.Path(ppt_path).stem}.parquet")
        df.to_parquet(out_path, index=False)
        return out_path, "fallback"
    except Exception as e:
        return None, f"fallback_error: {e}"

In [ ]:
# @title Certification processing that outputs CSV with 'yes'/'no' instead of numeric zeros
import pandas as pd
import os
import re
from typing import List

# Attempt to reuse the working DataFrame set by the integration cell
_df_names_priority: List[str] = [
    "df_main", "final_df", "results_df", "df", "dataframe", "output_df"
]

df = None
for name in _df_names_priority:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        df = globals()[name]
        break

if df is None:
    # Snapshot globals to avoid RuntimeError when the dict changes during iteration
    for name, val in list(globals().items()):
        try:
            if isinstance(val, pd.DataFrame):
                df = val
                break
        except Exception:
            pass

# Fallback: try to read an input CSV if present in the working directory
if df is None:
    for fname in ("input.csv", "data.csv", "dataset.csv"):
        if os.path.exists(fname):
            df = pd.read_csv(fname)
            break

if df is None:
    print("Skip: No DataFrame found. Define a DataFrame in previous cells or place 'input.csv' in the working directory.")
else:
    # Identify certification-related columns heuristically
    cert_cols = [c for c in df.columns if re.search(r"cert", str(c), re.IGNORECASE)]
    if not cert_cols:
        # If no obvious columns, attempt to expand search to common labels
        common_cert_labels = [
            "certifications", "certification", "certs", "cert_list", "has_cert", "is_certified"
        ]
        cert_cols = [c for c in df.columns if str(c).lower() in common_cert_labels]

    if not cert_cols:
        print("Skip: No certification-related columns found. Ensure your DataFrame includes columns like 'certifications' or contains 'cert'.")
    else:
        # Normalize certification presence: 'yes' if present, else 'no'
        normalized = pd.DataFrame(index=df.index)
        def to_yes_no(x):
            if pd.isna(x):
                return "no"
            # Treat lists/sets/tuples as present if non-empty
            if isinstance(x, (list, set, tuple)):
                return "yes" if len(x) > 0 else "no"
            # Strings: non-empty and not '0'/'none'
            if isinstance(x, str):
                s = x.strip().lower()
                if s in ("", "0", "none", "null", "false", "no"):
                    return "no"
                return "yes"
            # Numbers: non-zero -> yes
            try:
                if isinstance(x, (int, float)):
                    return "yes" if x != 0 else "no"
            except Exception:
                pass
            # Fallback: truthy -> yes
            return "yes" if bool(x) else "no"
        for c in cert_cols:
            normalized[c] = df[c].map(to_yes_no)

        # Also include a unified 'has_certification'
        normalized["has_certification"] = (
            normalized[cert_cols]
            .apply(lambda row: "yes" if any(val == "yes" for val in row) else "no", axis=1)
        )

        # Write output CSV
        out_path = "certification_output.csv"
        normalized.to_csv(out_path, index=False)

        # Display a small summary and the first few rows
        print(f"Wrote certification presence CSV to: {out_path}")
        print("Columns processed:", cert_cols)
        print("Sample output:")
        print(normalized.head().to_string(index=False))

Skip: No DataFrame found. Define a DataFrame in previous cells or place 'input.csv' in the working directory.


In [ ]:
# @title Integration: normalize certification fields and add unified yes/no column for frontend/export
import pandas as pd
import re
import os
from typing import List

# Configure the names your pipeline uses for the working DataFrame
_df_names_priority: List[str] = [
    "df_main", "final_df", "results_df", "df", "dataframe", "output_df"
]

# Find or create the working DataFrame
working_df_name = None
for name in _df_names_priority:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        working_df_name = name
        break

if working_df_name is None:
    # As a fallback, pick the first DataFrame in globals, snapshotting items to avoid RuntimeError
    for k, v in list(globals().items()):
        if isinstance(v, pd.DataFrame):
            working_df_name = k
            break

# If still none, try to load from a common CSV file so the pipeline can proceed
if working_df_name is None:
    for fname in ("input.csv", "data.csv", "dataset.csv"):
        if os.path.exists(fname):
            globals()["df_main"] = pd.read_csv(fname)
            working_df_name = "df_main"
            print(f"Loaded DataFrame from {fname} into 'df_main'.")
            break

if working_df_name is None:
    print("Skip: No DataFrame found or CSV to load. Define a DataFrame in previous cells (e.g., df_main) or place 'input.csv' in the working directory.")
else:
    df = globals()[working_df_name]

    # Detect certification-related columns
    cert_cols = [c for c in df.columns if re.search(r"cert", str(c), re.IGNORECASE)]
    if not cert_cols:
        # Consider common column names if 'cert' isn't present
        common_cert_labels = [
            "certifications", "certification", "certs", "cert_list", "has_cert", "is_certified"
        ]
        cert_cols = [c for c in df.columns if str(c).lower() in common_cert_labels]

    # Create/normalize a unified presence column: 'has_certification' -> 'yes'/'no'
    if len(cert_cols) == 0:
        # If no specific columns exist, create a default 'has_certification' as 'no'
        df["has_certification"] = "no"
        print("No certification columns found; added 'has_certification' with default 'no'.")
    else:
        def present(x):
            if pd.isna(x):
                return False
            if isinstance(x, (list, set, tuple)):
                return len(x) > 0
            if isinstance(x, str):
                s = x.strip().lower()
                return s not in ("", "0", "none", "null", "false", "no")
            if isinstance(x, (int, float)):
                return x != 0
            return bool(x)
        # Row-wise presence across all cert columns
        df["has_certification"] = (
            df[cert_cols]
            .apply(lambda row: any(present(val) for val in row), axis=1)
            .map(lambda b: "yes" if b else "no")
        )
        # Also convert individual cert columns to yes/no for clarity
        for c in cert_cols:
            df[c] = df[c].map(lambda x: "yes" if present(x) else "no")
        print(f"Normalized certification columns: {cert_cols} and added 'has_certification'.")

    # Update the global to ensure downstream cells (frontend/export) see the change
    globals()[working_df_name] = df
    print(f"Certification normalized on DataFrame '{working_df_name}'.")

Skip: No DataFrame found or CSV to load. Define a DataFrame in previous cells (e.g., df_main) or place 'input.csv' in the working directory.


In [ ]:
#@title frontend

import os, re, json, glob, math, textwrap, pathlib, inspect, shutil, tempfile, re, threading, time, warnings
from pathlib import Path
from typing import Optional, Dict, Any
import pandas as pd
import gradio as gr

# Silence noisy but harmless PIL warning
warnings.filterwarnings(
    "ignore",
    message="Palette images with Transparency expressed in bytes should be converted to RGBA images",
)

MASTER_SLIDES_PARQUET  = "master_slides.parquet"
EVAL_CSV_NAME          = "evaluation_results.csv"
EVAL_PARQUET_NAME      = "evaluation_results.parquet"
THEME_TRACK_PARQUET    = "theme_track_tech.parquet"

_AGG_FILES = {
    MASTER_SLIDES_PARQUET,
    EVAL_PARQUET_NAME,
    EVAL_CSV_NAME,
}

# Watsonx default models (unchanged; deprecation warnings may appear, but kept as requested)
WATSONX_DEFAULT_MODELS = [
    "openai/gpt-oss-120b",
    "ibm/granite-3-2-8b-instruct",
    "meta-llama/llama-3-3-70b-instruct",
    "ibm/granite-4-h-small",
]

# Hugging Face preset models when provider = "Hugging Face"
HF_PRESET_MODELS = [
    "mistralai/Mistral-7B-Instruct-v0.3",
    "ibm-granite/granite-3.3-8b-instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
]

# Default dimension & section weights
DEFAULT_DIM_ROWS = [
    ["Problem Statement", 0.10],
    ["Proposed Solution", 0.15],
    ["Technical Architecture", 0.15],
    ["Novelty  Uniqueness/creativity", 0.15],
    ["presentation", 0.10],
    ["ethical consideration", 0.10],
    ["completeness of design solution", 0.15],
    ["implementing ibm dpk/rag/aws/watsonx/granite", 0.10],
]
DEFAULT_SEC_ROWS = [
    ["problem_statement", 0.30],
    ["proposed_solution", 0.35],
    ["technical_architecture", 0.35],
]

# Detailed evaluation criterion columns
CRITERION_COLUMNS = [c[0] for c in DEFAULT_DIM_ROWS]

DETAILED_CRITERIA = {
    "Problem Statement": [
        "Is the problem statement clearly defined and realistic?",
        "Is the problem well-articulated and meaningful?"
    ],
    "Proposed Solution": [
        "Is the proposed solution comprehensive and well-designed?",
        "Does the solution effectively address the problem statement?"
    ],
    "Technical Architecture": [
        "Is the technical architecture clearly presented?",
        "Are system components and their interactions explained?",
        "Is the technology stack appropriate for the solution?",
        "Are technical diagrams present and clear?"
    ],
    "Novelty  Uniqueness/creativity": [
        "Is the technology suggested feasible to implement?",
        "Is there a working prototype?",
        "How have you tested your solution for any bias?",
        "What is unique/original/creative in this solution?"
    ],
    "presentation": [
        "Is the presentation clear, concise and structured?",
        "Is 'Why', 'What', and 'How' explained?",
        "Are services used clearly mentioned?",
        "Is the LLM used specified?",
        "Is training content used described?",
        "Is the architecture explained?"
    ],
    "ethical consideration": [
        "Is it ethical to implement the idea?",
        "Have you considered factors like gender neutrality?",
        "Are racial or religious biases being avoided?",
        "What bias testing methodology was used?",
        "How does the solution consider ethical implications?"
    ],
    "completeness of design solution": [
        "How is the solution making things better?",
        "What sort of deviation (positive/negative) can be seen by adopting your solution?",
        "Will it be scalable? Where else can this be used to make an impact?",
        "What can your solution do that just a simple Google search does not do?",
        "How would end-users use it?",
        "What is the potential positive impact on the chosen theme?"
    ],
    "implementing ibm dpk/rag/aws/watsonx/granite": [
        "Have you used one of the must-haves in your solution?",
        "What are the must-have techs that have been used in your solution?",
        "If using Traditional AI, is DPK usage present (mandatory)?",
        "How different is the solution compared to other LLMs being used?",
        "How is this solution better than using LLMs/Gen AI?",
        "Is IBM or AWS or both mentioned and used?",
        "Is RAG/Agentic usage clearly demonstrated?",
        "Are DPK/IBM Granite models used (RAG/Agentic Usage)?"
    ]
}

# Tick-based technology detection config
_TECH_TERMS = ["Data Prep Kit", "Granite", "RAG", "Agentic"]
TICK_CHARS = set(list("✓✔☑✅🗹🗸√"))
YES_PATTERN = re.compile(r"\byes\b", re.I)
NO_PATTERN = re.compile(r"\bno\b", re.I)

DEFAULT_DIM_JSON = json.dumps(DEFAULT_DIM_ROWS, indent=2)
DEFAULT_SECW_JSON = json.dumps(DEFAULT_SEC_ROWS, indent=2)

WX_URL_DEFAULT = os.environ.get("WATSONX_URL", "")
WX_KEY_DEFAULT = os.environ.get("WATSONX_API_KEY", "")
WX_PROJECT_DEFAULT = os.environ.get("WATSONX_PROJECT_ID", "")
HF_TOKEN_DEFAULT = os.environ.get("HUGGING_FACE_HUB_TOKEN", os.environ.get("HF_API_TOKEN", ""))

def _apply_hf_token(token: str) -> bool:
    tok = (token or "").strip()
    if not tok:
        return False
    try:
        os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
        os.environ["HF_API_TOKEN"] = tok
        try:
            from huggingface_hub import set_access_token
            try:
                set_access_token(tok)
            except Exception:
                from huggingface_hub import HfFolder
                HfFolder.save_token(tok)
        except Exception:
            pass
        return True
    except Exception:
        return False

def _ensure_dir(p): os.makedirs(p, exist_ok=True)
def _list_parquets(dir_path: str | Path) -> list[str]:
    p = Path(dir_path)
    return sorted([str(x) for x in p.glob("*.parquet")])
def _parse_csv_line(s: str) -> list[str]:
    return [x.strip() for x in (s or "").split(",") if x.strip()]
def _build_theme_dict(name: str, desc: str, kw: str, must: str = "", forb: str = "") -> dict:
    theme = {"name": (name or "").strip() or "Custom Theme", "description": (desc or "").strip(),
             "keywords": _parse_csv_line(kw), "must_haves": _parse_csv_line(must), "forbidden": _parse_csv_line(forb)}
    if not any([theme["description"], theme["keywords"], theme["must_haves"], theme["forbidden"]]): return {}
    return theme
def _chunked(iterable, n):
    for i in range(0, len(iterable), n):
        yield iterable[i:i+n]
def _unique_parquet_path(out_dir: Path, base_stem: str) -> Path:
    candidate = out_dir / f"{base_stem}.parquet"
    if not candidate.exists(): return candidate
    k = 2
    while True:
        candidate = out_dir / f"{base_stem}_v{k}.parquet"
        if not candidate.exists(): return candidate
        k += 1

def _fallback_extract_single(ppt_path: str, out_dir: str):
    try:
        from pptx import Presentation
        prs = Presentation(ppt_path)
        rows = []
        for idx, slide in enumerate(prs.slides, start=1):
            title_text = ""
            for shape in slide.shapes:
                if hasattr(shape, "text_frame") and shape.text_frame and shape.text_frame.text:
                    title_text = (shape.text_frame.text or "").strip()
                    if title_text: break
            body_parts = []
            for shape in slide.shapes:
                if hasattr(shape, "text_frame") and shape.text_frame:
                    txt = (shape.text_frame.text or "").strip()
                    if txt: body_parts.append(txt)
            rows.append({"slide_no": idx, "slide_title": title_text, "slide_content": "\n".join(body_parts),
                         "has_image": False, "extracted_content_from_image": "", "image_path": ""})
        df = pd.DataFrame(rows)
        _ensure_dir(out_dir)
        out_path = os.path.join(out_dir, f"{pathlib.Path(ppt_path).stem}.parquet")
        df.to_parquet(out_path, index=False)
        return out_path, "fallback"
    except Exception as e:
        return None, f"fallback_error: {e}"

def _append_df_to_parquet(path: Path, df: pd.DataFrame):
    """
    Append slide-rows into a master parquet. Dedupes safely.
    Requires at least slide_number. Uses submission_id when available.
    """
    if not isinstance(df, pd.DataFrame) or df.empty:
        return

    df = df.copy()

    # Make sure slide_number exists (best effort)
    if "slide_number" not in df.columns:
        # try common aliases
        for alt in ["slide_no", "slide_index", "page_number", "page_no"]:
            if alt in df.columns:
                df["slide_number"] = df[alt]
                break
    if "slide_number" not in df.columns:
        # cannot dedupe reliably; just write/append raw
        slide_subset = None
    else:
        slide_subset = ["slide_number"]

    # Ensure submission_id exists if possible (fallback to source_parquet stem)
    if "submission_id" not in df.columns:
        if "source_parquet" in df.columns:
            try:
                df["submission_id"] = df["source_parquet"].astype(str).str.replace(".parquet", "", regex=False)
            except Exception:
                df["submission_id"] = "unknown"
        else:
            df["submission_id"] = "unknown"

    # Preferred dedupe keys
    subset = ["submission_id"]
    if slide_subset:
        subset += slide_subset

    if path.exists():
        existing_df = pd.read_parquet(path)
        updated_df = pd.concat([existing_df, df], ignore_index=True)
        try:
            updated_df = updated_df.drop_duplicates(subset=subset, keep="last")
        except Exception:
            # If something is weird, just skip dedupe instead of crashing
            pass
    else:
        updated_df = df

    updated_df.to_parquet(path, index=False)

def _append_df_to_csv(path: Path, df: pd.DataFrame):
    if path.exists():
        existing_df = pd.read_csv(path)
        updated_df = pd.concat([existing_df, df]).drop_duplicates(subset=['submission_id'], keep='last')
    else:
        updated_df = df
    updated_df.to_csv(path, index=False)


def _ui_run_dpk_batch(ppt_paths, out_dir):
    logs, out_parquets = [], []
    out_root_path = Path(out_dir).resolve()
    _ensure_dir(out_root_path)
    extract_ppt_fn = globals().get("extract_ppt_with_dpk", None)
    extract_pdf_fn = globals().get("extract_pdf_to_parquet", None)

    for p in (ppt_paths or []):
        p_obj = Path(getattr(p, "name", p)).resolve()
        base = p_obj.name
        ext = p_obj.suffix.lower()
        logs.append(f" Extracting: {base}")
        try:
            pq_path = None
            if ext == ".pdf" and callable(extract_pdf_fn):
                pq_path = extract_pdf_fn(p_obj, out_root_path)
                logs.append("   ↗ handled as PDF (page→slide)")
            else:
                if callable(extract_ppt_fn):
                    try:
                        ret = extract_ppt_fn(p_obj, out_root_path)
                        pq_path = ret[0] if isinstance(ret, (tuple, list)) else ret
                        logs.append("   ↗ called extract_ppt_with_dpk(Path, Path)")
                    except TypeError as te1:
                        logs.append(f"   ↗ signature (Path, Path) failed: {te1}")
                        try:
                            ret = extract_ppt_fn(p_obj, out_root=out_root_path)
                            pq_path = ret[0] if isinstance(ret, (tuple, list)) else ret
                            logs.append("   ↗ called extract_ppt_with_dpk(out_root=Path)")
                        except TypeError as te2:
                            logs.append(f"   ↗ out_root kw failed: {te2}")
                            try:
                                ret = extract_ppt_fn(p_obj, out_dir=out_root_path)
                                pq_path = ret[0] if isinstance(ret, (tuple, list)) else ret
                                logs.append("   ↗ called extract_ppt_with_dpk(out_dir=Path)")
                            except Exception as te3:
                                logs.append(f"   ↗ out_dir kw failed: {te3}")
                                raise
                else:
                    pq_path, tag = _fallback_extract_single(str(p_obj), str(out_root_path))
                    logs.append(f"   ↗ fallback used ({tag})")

            if not pq_path:
                logs.append(f"Failed: {base} (no pq returned)")
                continue
            pq_path = str(Path(pq_path).resolve())
            if os.path.exists(pq_path):
                logs.append(f"{base} → {pq_path}")
                out_parquets.append(pq_path)
            else:
                logs.append(f"Failed: {base} (path does not exist: {pq_path})")
        except Exception as e:
            logs.append(f"Failed: {base} ({e})")

    if not out_parquets:
        globbed = _list_parquets(out_root_path)
        if globbed:
            logs.append(f"{len(globbed)} parquet(s) in {out_root_path}")
            out_parquets = globbed
        else:
            logs.append(f"No parquets found in {out_root_path}")
    return out_parquets, logs

# ------------------ Simple thread-safe logger ------------------
class LiveLogger:
    def __init__(self):
        self.lines = []
        self._lock = threading.Lock()
    def log(self, msg: str):
        ts = time.strftime('%H:%M:%S')
        with self._lock:
            self.lines.append(f"[{ts}] {msg}")
    def text(self):
        with self._lock:
            return "\n".join(self.lines[-800:])

eval_logger = LiveLogger()

# ------------------ Helpers and tech detection ------------------

def _rows_to_weight_dict(rows, key_field: str, weight_field: str) -> dict:
    out = {}
    if rows is None:
        return out
    if isinstance(rows, pd.DataFrame):
        rows_iter = rows.to_dict(orient="records")
    else:
        rows_iter = rows
    for r in rows_iter:
        if isinstance(r, dict):
            k = str(r.get(key_field, "")).strip()
            if not k:
                continue
            try:
                w = float(r.get(weight_field, 0))
            except Exception:
                w = 0.0
        elif isinstance(r, (list, tuple)) and len(r) >= 2:
            k = str(r[0]).strip()
            if not k:
                continue
            try:
                w = float(r[1])
            except Exception:
                w = 0.0
        else:
            continue
        out[k] = w
    total = sum(out.values())
    if total > 0:
        out = {k: v / total for k, v in out.items()}
    return out

try:
    from pptx import Presentation as _PPTX_Presentation
    from pptx.enum.shapes import MSO_SHAPE_TYPE as _PPTX_SHAPE_TYPE
except Exception:
    _PPTX_Presentation = None
    _PPTX_SHAPE_TYPE = None

def _shape_text(sh):
    try:
        if hasattr(sh, "text_frame") and sh.text_frame:
            return (sh.text_frame.text or "").strip()
        if getattr(sh, "text", None):
            return (sh.text or "").strip()
    except Exception:
        return ""
    return ""

def _row_bbox(table_shape, row_idx: int):
    top = table_shape.top
    for i in range(row_idx):
        top += table_shape.table.rows[i].height
    height = table_shape.table.rows[row_idx].height
    left = table_shape.left
    width = table_shape.width
    return left, top, width, height

def _point_in_row(x, y, row_bbox):
    left, top, width, height = row_bbox
    return left <= x <= left + width and top <= y <= top + height

def _is_small_icon(sh, row_height):
    try:
        if _PPTX_SHAPE_TYPE and sh.shape_type == _PPTX_SHAPE_TYPE.PICTURE:
            return sh.height <= row_height * 1.2 and sh.width <= row_height * 1.2
        if hasattr(sh, "width") and hasattr(sh, "height") and not _shape_text(sh):
            return sh.height <= row_height * 1.2 and sh.width <= row_height * 1.2
    except Exception:
        return False
    return False

def _extract_selected_technologies_from_tables(ppt_path: str):
    if _PPTX_Presentation is None:
        return False, []
    try:
        prs = _PPTX_Presentation(ppt_path)
    except Exception:
        return False, []
    selected = set(); any_table = False
    for slide in prs.slides:
        shapes_list = [s for s in slide.shapes]
        for table_shape in shapes_list:
            if not getattr(table_shape, "has_table", False):
                continue
            any_table = True
            table = table_shape.table
            row_bboxes = [_row_bbox(table_shape, r) for r in range(len(table.rows))]
            for r_idx, row in enumerate(table.rows):
                cell_texts = []
                for c in row.cells:
                    t = (c.text or "").strip()
                    if t: cell_texts.append(t)
                row_text_joined = " | ".join(cell_texts); row_low = row_text_joined.lower()
                tech_name = None
                for tech in _TECH_TERMS:
                    if re.search(rf"\b{re.escape(tech.lower())}\b", row_low):
                        tech_name = tech; break
                if not tech_name: continue
                tick_found = False
                if any(any(ch in TICK_CHARS for ch in ct) for ct in cell_texts):
                    tick_found = True
                if not tick_found:
                    yes_present = any(YES_PATTERN.search(t or "") for t in cell_texts)
                    no_present = any(NO_PATTERN.search(t or "") for t in cell_texts)
                    if yes_present and not no_present:
                        tick_found = True
                if not tick_found:
                    left, top, width, height = row_bboxes[r_idx]
                    for sh in shapes_list:
                        if sh == table_shape: continue
                        try:
                            sh_left, sh_top = sh.left, sh.top
                            sh_w, sh_h = getattr(sh, "width", 0), getattr(sh, "height", 0)
                            sh_cx = sh_left + sh_w / 2; sh_cy = sh_top + sh_h / 2
                            if _point_in_row(sh_cx, sh_cy, row_bboxes[r_idx]):
                                txt = _shape_text(sh)
                                if txt and any(ch in TICK_CHARS for ch in txt):
                                    tick_found = True; break
                                if txt and YES_PATTERN.search(txt) and not NO_PATTERN.search(txt):
                                    tick_found = True; break
                                if _is_small_icon(sh, height):
                                    tick_found = True; break
                        except Exception:
                            continue
                if tick_found:
                    selected.add(tech_name)
    return any_table, sorted(selected)

def _classify_submission(ppt_path: str) -> dict:
    p = Path(ppt_path)
    if not p.exists():
        return {"submission_id": p.stem, "technologies_used": "None", "technologies_status": "File Not Found"}
    any_table, selected = _extract_selected_technologies_from_tables(ppt_path)
    if not any_table:
        return {"submission_id": p.stem, "technologies_used": "None", "technologies_status": "No Table Found"}
    if not selected:
        return {"submission_id": p.stem, "technologies_used": "None", "technologies_status": "No Technologies Selected"}
    return {"submission_id": p.stem, "technologies_used": ", ".join(selected), "technologies_status": "Success"}

def _process_tech_batch_auto(ppt_paths, out_dir: str) -> pd.DataFrame:
    rows = [_classify_submission(p) for p in (ppt_paths or [])]
    df_new = pd.DataFrame(rows)
    scores_dir = Path(out_dir) / "scores"
    scores_dir.mkdir(parents=True, exist_ok=True)
    out_parquet = scores_dir / "theme_track_tech.parquet"
    if out_parquet.exists():
        try:
            old = pd.read_parquet(out_parquet)
            df = pd.concat([old, df_new], ignore_index=True).drop_duplicates("submission_id", keep="last")
        except Exception:
            df = df_new
    else:
        df = df_new
    try:
        df.to_parquet(out_parquet, index=False)
    except Exception:
        pass
    try:
        csv_path = scores_dir / "technologies_detection.csv"
        if csv_path.exists():
            oldc = pd.read_csv(csv_path)
            merged = pd.concat([oldc, df_new], ignore_index=True).drop_duplicates("submission_id", keep="last")
            merged.to_csv(csv_path, index=False)
        else:
            df_new.to_csv(csv_path, index=False)
    except Exception:
        pass
    return df

def _run_tech_extraction_auto(ppt_paths, out_dir):
    # Filter only PPT/PPTX for table-tick detection (PDFs are N/A)
    ppt_only = [p for p in (ppt_paths or []) if str(p).lower().endswith((".ppt",".pptx"))]
    try:
        df = _process_tech_batch_auto(ppt_only, out_dir)
    except Exception:
        df = pd.DataFrame()
    wanted = [c for c in ("submission_id","technologies_used","technologies_status") if c in df.columns]
    return df[wanted] if wanted else df

# Backwards-compat alias
def process_tech_batch_auto(ppt_paths, out_parquet: str) -> pd.DataFrame:
    try:
        out_dir = str(Path(out_parquet).parent)
    except Exception:
        out_dir = "."
    ppt_only = [p for p in (ppt_paths or []) if str(p).lower().endswith((".ppt",".pptx"))]
    df = _process_tech_batch_auto(ppt_only, out_dir)
    try:
        Path(out_parquet).parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(out_parquet, index=False)
    except Exception:
        pass
    return df

def _merge_technologies(scores_df: pd.DataFrame, tech_df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(scores_df, pd.DataFrame) or scores_df.empty:
        return scores_df if isinstance(scores_df, pd.DataFrame) else pd.DataFrame()
    if isinstance(tech_df, pd.DataFrame) and not tech_df.empty and 'submission_id' in scores_df.columns and 'submission_id' in tech_df.columns:
        try:
            return scores_df.merge(tech_df, on='submission_id', how='left', suffixes=("", "_tech"))
        except Exception:
            return scores_df
    return scores_df

def _serialize_feedback_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty: return df
    df = df.copy()

    # Remove certificate-related columns from display
    cert_cols = [c for c in df.columns if 'certificate' in c.lower()]
    df = df.drop(columns=cert_cols, errors='ignore')

    feedback_cols = [c for c in df.columns if 'feedback' in c.lower()]
    for c in feedback_cols:
        def _dump(v):
            if pd.isna(v): return v
            if isinstance(v, (dict, list)):
                try: return json.dumps(v, ensure_ascii=False)
                except: return str(v)
            return str(v)
        df[c] = df[c].apply(_dump)
        preview_col = c + "_preview" if not c.endswith("_preview") else c
        def _preview(s):
            if pd.isna(s): return s
            s = str(s)
            if len(s) <= 300: return s
            return s[:300].rstrip() + " …"
        df[preview_col] = df[c].apply(_preview)
    if 'feedback' not in df.columns and feedback_cols:
        df['feedback'] = df[feedback_cols[0]]
    if 'feedback_preview' not in df.columns and feedback_cols:
        src_preview = feedback_cols[0] + "_preview" if not feedback_cols[0].endswith("_preview") else feedback_cols[0]
        df['feedback_preview'] = df.get(src_preview, "")
    return df

def _normalize_final_score_column(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty: return df
    df = df.copy()
    if 'final_score' in df.columns and df['final_score'].notna().any():
        return df
    candidates = [c for c in df.columns if re.search(r'final.*score|score.*final|final_.*_score', c, flags=re.I)]
    if not candidates:
        candidates = [c for c in df.columns if c.lower().endswith("_0_100") and 'final' in c.lower()]
    if not candidates:
        for alt in ['final', 'score', 'final_0_100', 'final_score_0_100', 'final_score_0_1']:
            if alt in df.columns:
                candidates = [alt]; break
    if candidates:
        pref = next((c for c in ['final_score_0_100','final_0_100','final_score'] if c in df.columns), candidates[0])
        df['final_score'] = df[pref]
    else:
        df['final_score'] = 0.0
    try:
        df['final_score'] = pd.to_numeric(df['final_score'], errors='coerce').fillna(0.0)
    except Exception:
        pass
    return df

def _ui_run_eval(parquet_paths, wx_url, wx_key, wx_project, models_csv, dim_json, secw_json, mode, cap, theme: Optional[Dict[str, Any]] = None, provider: str = "Watsonx", hf_api_token: str = ""):
    run_eval = globals().get("run_evaluation", None)
    if not callable(run_eval):
        return "run_evaluation(...) is not defined in this environment. Please load it.", pd.DataFrame()

    # Apply HF token early if needed
    if str(provider).lower().startswith("hugging"):
        _apply_hf_token(hf_api_token)

    try:
        models = [m.strip() for m in (models_csv or "").split(",") if m.strip()]
    except:
        models = []
    try:
        dims = json.loads(dim_json) if dim_json else {}
    except:
        dims = {}
    try:
        secw = json.loads(secw_json) if secw_json else {}
    except:
        secw = {}

    parquet_paths = [str(p) for p in (parquet_paths or []) if isinstance(p, (str, Path))]

    desired_kwargs = dict(
        input_dir=".",
        watsonx_url=wx_url,
        watsonx_api_key=wx_key,
        watsonx_project_id=wx_project,
        models_csv=",".join(models) if models else "",
        dimensions_json=json.dumps(dims) if dims else "",
        section_weights_json=json.dumps(secw) if secw else "",
        certificate_weight=0.0,  # Keep in backend but set to 0.0
        best_of_mode=mode,
        cap_pct=float(cap),
        theme=theme,
        out_prefix="scores/run",
        provider=provider,
        hf_api_token=hf_api_token
    )

    sig = inspect.signature(run_eval)
    accepted = set(sig.parameters.keys())
    filtered = {k: v for k, v in desired_kwargs.items() if k in accepted}

    wants_input_dir = 'input_dir' in accepted
    tmp_dir = None
    if wants_input_dir and parquet_paths:
        tmp_dir = tempfile.mkdtemp(prefix="flat_eval_")
        for p in parquet_paths:
            try:
                df_original = pd.read_parquet(p)
                dest = os.path.join(tmp_dir, os.path.basename(p))
                df_original.to_parquet(dest, index=False)
            except Exception as e:
                print(f"Warning: Could not copy {p} to temporary directory {tmp_dir}: {e}")
        filtered['input_dir'] = tmp_dir

    try:
        try:
            result = run_eval(**filtered)
        except TypeError:
            filtered2 = {k: v for k, v in filtered.items() if k in accepted}
            result = run_eval(**filtered2)
    finally:
        if tmp_dir and os.path.exists(tmp_dir):
            try:
                shutil.rmtree(tmp_dir)
            except Exception as e:
                print(f"Warning: Could not remove temporary directory {tmp_dir}: {e}")

    if isinstance(result, tuple) and len(result) == 2:
        logs, top = result
    elif isinstance(result, pd.DataFrame):
        logs, top = "Evaluation completed.", result
    else:
        logs, top = str(result), pd.DataFrame()

    top = _normalize_final_score_column(top)
    top = _serialize_feedback_columns(top)

    footer = f"\n\n(Found {len(parquet_paths)} parquet(s); "
    if wants_input_dir and tmp_dir:
        footer += f"flattened into {tmp_dir} for evaluator expecting input_dir.)"
    else:
        footer += "passed as explicit file list.)"
    logs = (logs or "") + footer

    top20 = top.head(20) if not top.empty else pd.DataFrame()
    return logs, top20

def _df_to_rows(df: pd.DataFrame):
    try:
        return df.reset_index(drop=True)
    except Exception:
        return pd.DataFrame()

def _extract(ppt_paths, out_dir):
    if not ppt_paths:
        return "No PPTs/PDFs selected.", [], out_dir, [], pd.DataFrame(), pd.DataFrame()
    out_dir_path = Path(out_dir).resolve(); _ensure_dir(out_dir_path)
    parquet_paths, logs1 = _ui_run_dpk_batch(ppt_paths, out_dir)
    rows_for_parquet_table = []
    logs2 = []

    for pq in parquet_paths:
        rows_for_parquet_table.append([pq])

        try:
            df_slides = pd.read_parquet(pq).copy()
            df_slides["source_parquet"] = os.path.basename(pq)
            append_func = globals().get("_append_df_to_parquet", None)
            if callable(append_func):
                append_func(out_dir_path/MASTER_SLIDES_PARQUET, df_slides)
            else:
                logs2.append(f"Could not append slides from {pq}: _append_df_to_parquet not found or not callable.")
        except Exception as e:
            logs2.append(f"Could not append slides from {pq}: {e}")

    try:
        res = globals().get("scan_missing_requirements", None)
        miss_df = pd.DataFrame(); miss_csv_path = out_dir_path/"scores"/"missing_requirements.csv"
        if callable(res):
            ret = res(out_dir_path)
            if isinstance(ret, tuple) and len(ret) == 2:
                a,b = ret
                if isinstance(a, pd.DataFrame): miss_df=a; miss_csv_path=Path(b) if not isinstance(b,Path) else b
                elif isinstance(b, pd.DataFrame): miss_df=b; miss_csv_path=Path(a) if not isinstance(a,Path) else a
            elif isinstance(ret, pd.DataFrame): miss_df=ret
        if miss_df.empty:
            logs2.append("Missing requirements: none missing across submissions")
        else:
            _ensure_dir(miss_csv_path.parent)
            try: miss_df.to_csv(miss_csv_path, index=False)
            except Exception: pass
            logs2.append(f"Missing requirements: wrote {miss_csv_path} with {len(miss_df)} rows")
    except Exception as e:
        miss_df = pd.DataFrame(); logs2.append(f"Missing requirements scan failed: {e}")

    try:
        fn = globals().get("_run_tech_extraction_auto")
        if callable(fn):
            tech_df = fn(ppt_paths, str(out_dir_path))
        else:
            tech_df = process_tech_batch_auto(ppt_paths, str(out_dir_path/THEME_TRACK_PARQUET))
    except Exception as e:
        logs2.append(f"Technology detection failed: {e}")
        tech_df = pd.DataFrame([["error", str(e), "fail"]], columns=["submission_id","technologies_used","technologies_status"])

    logs = "\n".join(logs1 + logs2) if (logs1 or logs2) else "No logs."

    return logs, rows_for_parquet_table, out_dir_path, parquet_paths, tech_df, tech_df


def _get_submission_id_from_parquet(path_str: str) -> str:
    p = Path(path_str)
    try:
        df = pd.read_parquet(p)
        for cand in ["submission_id","file_stem","name"]:
            if cand in df.columns:
                v = df[cand].iloc[0]
                if isinstance(v,str) and v.strip(): return v.strip()
    except Exception:
        pass
    return p.stem

def _evaluate(parquets, out_dir_str, url, key, project, models, dj, sj, mode, cap,
              drop_key, use_custom_val, name, desc, kw, must, forb, themes_state,
              skip_flag, missing_df):
    try:
        if use_custom_val: theme_obj = _build_theme_dict(name, desc, kw, must, forb)
        else: theme_obj = (themes_state or {}).get(drop_key or "", {})
        if theme_obj == {}: theme_obj = None
    except Exception:
        theme_obj = None

    collected = list(parquets or [])
    cand_dirs = set()
    if out_dir_str and os.path.isdir(out_dir_str): cand_dirs.add(str(Path(out_dir_str).resolve()))
    for p in collected:
        try:
            pth = Path(p).resolve()
            cand_dirs.add(str(pth.parent)); cand_dirs.add(str(pth.parent.parent))
        except Exception: pass
    for d in list(cand_dirs):
        for root, _, files in os.walk(d):
            for fn in files:
                if fn.lower().endswith(".parquet") and fn not in _AGG_FILES:
                    collected.append(os.path.join(root, fn))
    collected = sorted(set(collected))
    if not collected:
        return "No parquets found (recursively) under the selected output directory. Run DPK first.", pd.DataFrame()

    missing_ids = set()
    try:
        mdf = pd.DataFrame(missing_df) if isinstance(missing_df,(list,dict,pd.DataFrame)) else pd.DataFrame()
        default_miss_csv = Path(out_dir_str or ".")/"scores"/"missing_requirements.csv"
        if mdf.empty and default_miss_csv.exists(): mdf = pd.read_csv(default_miss_csv)
        if not mdf.empty and "missing_any" in mdf.columns and "submission_id" in mdf.columns:
            missing_ids = set(mdf.loc[mdf["missing_any"]==True, "submission_id"].astype(str))
    except Exception:
        missing_ids = set()

    if skip_flag and missing_ids:
        filtered=[]
        for p in collected:
            sid = _get_submission_id_from_parquet(p)
            if sid in missing_ids: continue
            filtered.append(p)
        collected = filtered
        if not collected:
            return "All submissions have missing required sections. Nothing to evaluate.", pd.DataFrame()

    logs_all = []; dfs_for_display = []

    logs, top = _ui_run_eval(collected, url, key, project, models, dj, sj, mode, cap, theme=theme_obj)
    logs_all.append(logs)

    if isinstance(top, pd.DataFrame) and not top.empty:
        df_res = top.copy()
        df_res = _normalize_final_score_column(df_res)
        df_res = _serialize_feedback_columns(df_res)

        _ensure_dir("scores")
        try:
            _append_df_to_csv = globals().get("_append_df_to_csv", None)
            if callable(_append_df_to_csv):
                _append_df_to_csv(Path("scores")/EVAL_CSV_NAME, df_res)
            else:
                logs_all.append("Could not append to CSV: _append_df_to_csv not found or not callable.")
        except Exception as e:
            logs_all.append(f"Could not append to CSV: {e}")
        try:
            _append_df_to_parquet = globals().get("_append_df_to_parquet", None)
            if callable(_append_df_to_parquet):
                _append_df_to_parquet(Path("scores")/EVAL_PARQUET_NAME, df_res)
            else:
                logs_all.append("Could not append to Parquet: _append_df_to_parquet not found or not callable.")
        except Exception as e:
            logs_all.append(f"Could not append to Parquet: {e}")
        dfs_for_display.append(df_res)

    if dfs_for_display:
        combined = pd.concat(dfs_for_display, ignore_index=True)
        combined = _normalize_final_score_column(combined)
        combined = _serialize_feedback_columns(combined)

        id_candidates = ["submission_id","file_stem","name","submission","id","source_parquet","parquet_path"]
        id_col = next((c for c in id_candidates if c in combined.columns), None)

        feedback_cols = [c for c in combined.columns if 'feedback' in c.lower()]
        final_variants = [c for c in combined.columns if re.search(r'final.*score|score.*final|final_.*_score', c, flags=re.I)]
        preferred = []
        if id_col: preferred.append(id_col)
        preferred.append('final_score')
        for v in final_variants:
            if v not in preferred and v in combined.columns: preferred.append(v)
        if 'feedback_preview' in combined.columns and 'feedback_preview' not in preferred:
            preferred.append('feedback_preview')
        for c in feedback_cols:
            if c not in preferred: preferred.append(c)
        if 'feedback' in combined.columns and 'feedback' not in preferred:
            preferred.append('feedback')

        ordered = []
        for c in preferred:
            if c in combined.columns and c not in ordered: ordered.append(c)
        for c in combined.columns.tolist():
            if c not in ordered:
                ordered.append(c)
        top20 = combined.sort_values(by="final_score", ascending=False).head(20)
        display_cols = [c for c in ordered if c in top20.columns]
        display_df = top20[display_cols]
    else:
        display_df = pd.DataFrame()

    if skip_flag and missing_ids:
        logs_all.append(f"\n(Skipped {len(missing_ids)} submission(s) with missing required sections.)")

    return "\n".join(logs_all), display_df

# Gradio Frontend
with gr.Blocks(title=" AI Evaluator") as demo:
    gr.Markdown("## AI Evaluator")

    st_out_dir = gr.State("pipeline_output")
    st_parquets = gr.State([])
    st_tech = gr.State(pd.DataFrame())
    st_dims = gr.State(DEFAULT_DIM_ROWS)
    st_secs = gr.State(DEFAULT_SEC_ROWS)
    st_themes = gr.State({"default": {"name": "Default Theme", "description": "", "keywords": []}})
    st_eval_full = gr.State(pd.DataFrame())

    with gr.Tab("1) Inputs"):
        drive_dir = gr.Textbox(label="Folder path to theme")
        scan_btn = gr.Button("Scan Folder")
        uploads = gr.Files(label="Upload .ppt/.pptx/.pdf", file_types=[".ppt", ".pptx", ".pdf"], type="filepath")
        out_parquet_dir = gr.Textbox(label="Parquet output directory", value="pipeline_output")
        ppt_table = gr.Dataframe(headers=["ppt_or_pdf_path"], datatype=["str"], interactive=False, label="PPT/PDF Files")
        ppt_state = gr.State([])

    with gr.Tab("2) Extraction"):
        run_extract = gr.Button("Run Extraction")
        extract_logs = gr.Textbox(label="Logs", lines=18)
        parquet_table = gr.Dataframe(headers=["parquet_path"], datatype=["str"], interactive=False, label="Parquets")
        classification_table = gr.Dataframe(label="Detected Technologies (PPT only)", interactive=False)

    with gr.Tab("3) Evaluation"):
        with gr.Row():
            model_checks = gr.CheckboxGroup(
                choices=WATSONX_DEFAULT_MODELS,
                value=WATSONX_DEFAULT_MODELS,
                label="Select Models"
            )
        with gr.Row():
            provider_radio = gr.Radio(choices=["Watsonx", "Hugging Face"], value="Watsonx", label="Inference Provider")

        with gr.Accordion("Watsonx Credentials (optional)", open=False):
            wx_url = gr.Textbox(label="WATSONX_URL", value=WX_URL_DEFAULT)
            wx_key = gr.Textbox(label="WATSONX_API_KEY", value=WX_KEY_DEFAULT, type="password")
            wx_project = gr.Textbox(label="WATSONX_PROJECT_ID", value=WX_PROJECT_DEFAULT)

        with gr.Accordion("Hugging Face Credentials (optional)", open=False):
            hf_token = gr.Textbox(label="HUGGING_FACE_HUB_TOKEN", value=HF_TOKEN_DEFAULT, type="password")

        with gr.Row():
            agg_mode = gr.Radio(choices=["average","max"], value="average", label="Aggregation Mode")

        with gr.Row(equal_height=True):
            th_name = gr.Textbox(label="Theme name", value="Custom Theme", lines=2)
            th_desc = gr.Textbox(label="Short description", value="", lines=2)

        theme_dropdown = gr.Dropdown(
            label="Saved Theme",
            choices=["default"],
            value="default",
            allow_custom_value=True
        )
        use_custom = gr.Checkbox(label="Use custom theme fields above", value=True)
        th_kw = gr.Textbox(label="Keywords")

        with gr.Row():
            save_theme_btn = gr.Button("Save / Update Theme")
            theme_preview = gr.Code(label="Theme JSON", value="{}", language="json")

        gr.Markdown("### Dimension Weights ")
        dim_table = gr.Dataframe(
            headers=["criteria","weight"], datatype=["str","number"],
            value=DEFAULT_DIM_ROWS, row_count=(len(DEFAULT_DIM_ROWS)+1, "dynamic"), col_count=2
        )
        gr.Markdown("### Section Weights ")
        sec_table = gr.Dataframe(
            headers=["section","weight"], datatype=["str","number"],
            value=DEFAULT_SEC_ROWS, row_count=(len(DEFAULT_SEC_ROWS)+1, "dynamic"), col_count=2
        )
        weight_warning = gr.Markdown("", elem_id="weight-warning")

        run_eval_btn = gr.Button("Run Evaluation ")
        eval_logs = gr.Textbox(label="Evaluation Logs (streaming)", lines=20)
        top20 = gr.Dataframe(label="Top 20 (Avg Final Score)", interactive=False)
        with gr.Row():
            detail_id = gr.Dropdown(label="Select Submission for Details", choices=[], value=None, interactive=True)
        with gr.Accordion("Submission Detail (Feedback & Missing Requirements)", open=False):
            detail_feedback = gr.Textbox(label="Feedback", lines=6, interactive=False)
            detail_missing = gr.Textbox(label="Missing Requirements", lines=4, interactive=False)

    # ------------------ Callbacks ------------------
    def _scan(folder, uploaded):
        paths=[]
        if folder and os.path.isdir(folder):
            paths.extend(sorted(
                glob.glob(os.path.join(folder, "*.pptx")) +
                glob.glob(os.path.join(folder, "*.ppt")) +
                glob.glob(os.path.join(folder, "*.pdf"))
            ))
        if uploaded:
            for f in uploaded:
                p=getattr(f, "name", f)
                if isinstance(p,str) and p.lower().endswith(('.ppt','.pptx','.pdf')):
                    paths.append(p)
        paths=sorted(set(paths))
        return [[p] for p in paths], paths, paths

    scan_btn.click(_scan, inputs=[drive_dir, uploads], outputs=[ppt_table, ppt_state, st_parquets])

    def _on_provider_change(provider):
        if provider == "Hugging Face":
            return gr.update(choices=HF_PRESET_MODELS, value=HF_PRESET_MODELS)
        else:
            return gr.update(choices=WATSONX_DEFAULT_MODELS, value=WATSONX_DEFAULT_MODELS)

    provider_radio.change(_on_provider_change, inputs=[provider_radio], outputs=[model_checks])

    run_extract.click(
        _extract,
        inputs=[ppt_state, out_parquet_dir],
        outputs=[extract_logs, parquet_table, st_out_dir, st_parquets, classification_table, st_tech],
        queue=True,
    )

    def _init_theme_choices(themes):
        names = sorted(list((themes or {}).keys())) or ["default"]
        _apply_hf_token(HF_TOKEN_DEFAULT)
        return gr.update(choices=names, value=names[0])
    demo.load(_init_theme_choices, inputs=[st_themes], outputs=[theme_dropdown])

    def _sum_weights(rows, weight_field: str) -> float:
        total = 0.0
        try:
            if isinstance(rows, pd.DataFrame):
                iterable = rows.to_dict(orient="records")
            else:
                iterable = rows or []
            for r in iterable:
                if isinstance(r, dict):
                    w = r.get(weight_field, None)
                elif isinstance(r, (list, tuple)) and len(r) >= 2:
                    w = r[1]
                else:
                    w = None
                try:
                    if w is not None and str(w).strip() != "":
                        total += float(w)
                except Exception:
                    continue
        except Exception:
            pass
        return total

    def _validate_weights_cb(dim_rows, sec_rows):
        dim_sum = _sum_weights(dim_rows, "weight")
        sec_sum = _sum_weights(sec_rows, "weight")
        tol = 1e-3
        ok = (abs(dim_sum - 1.0) <= tol) and (abs(sec_sum - 1.0) <= tol)
        if ok:
            msg = ""
        else:
            msg = f"⚠️ Weights must sum to 1.0. Current sums — Dimensions: {dim_sum:.3f}, Sections: {sec_sum:.3f}."
        return gr.update(interactive=ok), gr.update(value=msg)

    dim_table.change(_validate_weights_cb, inputs=[dim_table, sec_table], outputs=[run_eval_btn, weight_warning])
    sec_table.change(_validate_weights_cb, inputs=[dim_table, sec_table], outputs=[run_eval_btn, weight_warning])

    def _evaluate_stream(parquets, out_dir, models, dim_rows, sec_rows, tech_df,
                         provider, wx_url_v, wx_key_v, wx_proj_v, hf_token_v,
                         agg_mode_v, theme_choice, use_custom_flag, name, desc, kw, themes_state):
        eval_logger.lines = []
        if not parquets:
            eval_logger.log("No parquets to evaluate.")
            yield eval_logger.text(), pd.DataFrame(), pd.DataFrame(), gr.update(choices=[], value=None), "", ""
            return
        selected = models or []
        if not selected:
            eval_logger.log("No models selected.")
            yield eval_logger.text(), pd.DataFrame(), pd.DataFrame(), gr.update(choices=[], value=None), "", ""
            return

        if str(provider).lower().startswith("hugging"):
            applied = _apply_hf_token(hf_token_v)
            eval_logger.log(f"HF token applied: {'yes' if applied else 'no'}")

        dim_w = _rows_to_weight_dict(dim_rows, "criteria", "weight")
        sec_w = _rows_to_weight_dict(sec_rows, "section", "weight")

        try:
            themes = themes_state or {}
            if use_custom_flag:
                theme_obj = _build_theme_dict(name, desc, kw)
            else:
                theme_obj = themes.get(theme_choice, {}) or _build_theme_dict(name, desc, kw)
        except Exception:
            theme_obj = {}

        eval_logger.log(f"Provider: {provider} | Models: {len(selected)} | Dim keys: {len(dim_w)} | Sec keys: {len(sec_w)} | Agg: {agg_mode_v} | Theme: {theme_obj.get('name','n/a')}")
        eval_logger.log(f"Parquet files to evaluate: {len(parquets)}")

        avg = None
        try:
            if provider == "Watsonx":
                avg_fn = globals().get('_average_across_models_credentials', None)
                if callable(avg_fn):
                    avg = avg_fn(parquets, selected, dim_w, sec_w, wx_url_v, wx_key_v, wx_proj_v, theme_obj, agg_mode_v, 0.0)
                else:
                    models_csv = ",".join(selected)
                    dim_json = json.dumps(dim_w)
                    secw_json = json.dumps(sec_w)
                    best_of_mode = True if str(agg_mode_v).lower() == "max" else False
                    _, avg = _ui_run_eval(parquets, wx_url_v, wx_key_v, wx_proj_v, models_csv, dim_json, secw_json, best_of_mode, 0.0, theme=theme_obj, provider="Watsonx", hf_api_token="")
            else:
                avg_hf = globals().get('_average_across_models_hf', None)
                if callable(avg_hf):
                    avg = avg_hf(parquets, selected, dim_w, sec_w, hf_token_v, theme_obj, agg_mode_v, 0.0)
                else:
                    models_csv = ",".join(selected)
                    dim_json = json.dumps(dim_w)
                    secw_json = json.dumps(sec_w)
                    best_of_mode = True if str(agg_mode_v).lower() == "max" else False
                    _, avg = _ui_run_eval(parquets, "", "", "", models_csv, dim_json, secw_json, best_of_mode, 0.0, theme=theme_obj, provider="Hugging Face", hf_api_token=hf_token_v)
        except Exception as e:
            eval_logger.log(f"Evaluator error ({provider}): {e}")
            avg = pd.DataFrame()
        if avg is None:
            avg = pd.DataFrame()

        # Merge technologies (PPT-only)
        if isinstance(tech_df, list):
            try:
                if tech_df and isinstance(tech_df[0], dict):
                    tech_df = pd.DataFrame(tech_df)
                else:
                    tech_df = pd.DataFrame(tech_df, columns=["submission_id","technologies_used","technologies_status"]) if tech_df else pd.DataFrame()
            except Exception:
                tech_df = pd.DataFrame()
        elif not isinstance(tech_df, pd.DataFrame):
            tech_df = pd.DataFrame()
        merged = _merge_technologies(avg, tech_df)
        score_col = "final_score" if "final_score" in merged.columns else ("final_score_avg" if "final_score_avg" in merged.columns else None)
        if not merged.empty and score_col:
            merged = merged.sort_values(score_col, ascending=False)

            # Hide certificate, feedback and missing columns for display
            hide_cols = {c for c in merged.columns if "feedback" in c.lower() or "missing" in c.lower() or "certificate" in c.lower()}
            summary_cols = [c for c in merged.columns if c not in hide_cols]
            top20_df = merged[summary_cols].head(20).copy()

            _ensure_dir("scores")
            top20_path = Path("scores") / "top20.csv"
            top20_df.to_csv(top20_path, index=False)
            eval_logger.log(f"Saved {top20_path} ({len(top20_df)} rows) using aggregation={agg_mode_v}.")
            sub_choices = merged["submission_id"].head(200).tolist() if "submission_id" in merged.columns else []
        else:
            top20_df = pd.DataFrame()
            eval_logger.log("No evaluation data produced.")
            sub_choices = []

        yield eval_logger.text(), top20_df, merged, gr.update(choices=sub_choices, value=None), "", ""

    def _show_details(sub_id, full_df):
        if not sub_id or isinstance(full_df, list) or not hasattr(full_df, 'columns'):
            return "", ""
        try:
            row = full_df[full_df['submission_id']==sub_id].iloc[0]
        except Exception:
            return "", ""
        fb = row.get('Feedback', '') or row.get('feedback','') or ''
        miss = row.get('Missing Requirements','') or row.get('missing_requirements','') or ''
        return str(fb), str(miss)

    run_eval_btn.click(
        _evaluate_stream,
        inputs=[st_parquets, st_out_dir, model_checks, dim_table, sec_table, st_tech,
                provider_radio, wx_url, wx_key, wx_project, hf_token,
                agg_mode, theme_dropdown, use_custom, th_name, th_desc, th_kw, st_themes],
        outputs=[eval_logs, top20, st_eval_full, detail_id, detail_feedback, detail_missing],
        queue=True,
    )
    detail_id.change(_show_details, inputs=[detail_id, st_eval_full], outputs=[detail_feedback, detail_missing])

    def _save_theme(name, desc, kw, themes):
        th = _build_theme_dict(name, desc, kw)
        themes = dict(themes or {})
        themes[th['name']] = th
        theme_names = sorted(list(themes.keys())) or ["default"]
        return themes, gr.update(choices=theme_names, value=th['name']), json.dumps(th, indent=2)

    def _preview_theme(drop_key, use_custom_flag, name, desc, kw, themes):
        if use_custom_flag:
            th = _build_theme_dict(name, desc, kw)
        else:
            all_themes = themes or {}
            if drop_key not in all_themes:
                if 'default' in all_themes:
                    th = all_themes['default']
                else:
                    th = _build_theme_dict(name, desc, kw)
            else:
                th = all_themes.get(drop_key, {})
        return json.dumps(th, indent=2)

    save_theme_btn.click(_save_theme, inputs=[th_name, th_desc, th_kw, st_themes], outputs=[st_themes, theme_dropdown, theme_preview])
    for comp in [theme_dropdown, use_custom, th_name, th_desc, th_kw]:
        comp.change(_preview_theme, inputs=[theme_dropdown, use_custom, th_name, th_desc, th_kw, st_themes], outputs=[theme_preview])

demo.launch(debug=True, share=False)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/ibm_watsonx_ai/foundation_models/utils/utils.py:428: LifecycleWarning: Model 'ibm/granite-3-2-8b-instruct' is in deprecated state from 2025-11-24 until 2026-02-22. IDs of alternative models: ibm/granite-4-h-small. Further details: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-model-lifecycle.html?context=wx&audience=wdp
  warn(model_state_warning, category=LifecycleWarning)
/usr/local/lib/python3.12/dist-packages/ibm_watsonx_ai/foundation_models/utils/utils.py:428: LifecycleWarning: Model 'ibm/granite-3-2-8b-instruct' is in deprecated state from 2025-11-24 until 2026-02-22. IDs of alternative models: ibm/granite-4-h-small. Further details: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-model-lifecycle.html?context=wx&audience=wdp
  warn(model_state_warning, category=LifecycleWarning)
/usr/local/lib/python3.12/dist-packages/ibm_watsonx_ai/foundation_models/utils/utils.py:428: LifecycleWarning: Model 'ibm/

In [ ]:
# @title Clean unnecessary things

# !rm -rf pipeline_output
# !rm -rf input_submissions
# !rm -rf scores
# !rm -rf results
# !rm -rf fetch1.csv

In [ ]:
# @title Converting Parquets to csv (only for test)

import pandas

pandas.read_parquet('/content/drive/MyDrive/hackathon_ppts/pipeline_output/Theme2 - CalmCraft -.pptx/Theme2 - CalmCraft -.pptx.parquet').to_csv('fetch1.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/hackathon_ppts/pipeline_output/Theme2 - CalmCraft -.pptx/Theme2 - CalmCraft -.pptx.parquet'